# Reproduction code: Identifying the viral and epidemiological factors behind the apparent global extinction of influenza B/Yamagata

This notebook reproduces every model result, figure and table of the paper and of its Supporting
Information (SI). It uses the two-lineage (Victoria, Yamagata), three-region influenza B model in
`simulator.py` (deterministic ODEs plus stochastic tau-leaping) and, for the SI section on the
all-or-none immunity assumption, the same model with leaky cross-immunity in `simulator_leaky.py`.
Run the cells from top to bottom; every output is written to this folder.

| Section | Output files | In the paper |
|---|---|---|
| 1. FluNet surveillance data | `nh_data.png`, `data.png` | Fig. 1, Fig. S1 |
| 2. Latin hypercube search | `search_1M.csv` (generated, not bundled) | input to Sections 3 and 4 |
| 3. Selection, biological filter, parameter space | `target_runs.csv`, `filtered.csv`, `both_outcomes.png`, `both_outcomes_pairs_green.png`, `both_outcomes_pairs_blue.png`, `shap_beeswarm.png`, `table_1.csv` | Table 1, Figs. S2–S5 |
| 4. Likelihood of the filtered samples | `filtered_outputs.csv`, `evaluation_densities.png`, `nh_fit_rank1.png`–`nh_fit_rank4.png`, `rho_distributions.png`, `rho_vs_parameters.png`, `rho_spearman.csv` | Fig. 2, Figs. S10, S11, S20–S23 |
| 5. Stochastic runs of the four best fits | `sto_nh.png`, `sto_nh_rank2.png`–`sto_nh_rank4.png`, `stochastic_victoria.csv`, `reintroduction.csv` | Fig. 3, Figs. S13, S15, S17 |
| 6. Yamagata's $R_e$ and outputs across the filtered samples | `Yam.png`, `Yam_dif.png`, `min_vs_r0_difference.png`, `Yam_scatter.png`, `extinction_vs_minimum.png`, `summary_across_samples.csv` | Fig. 4, Figs. S6–S9 |
| 7. Leaky cross-immunity | `leaky_scan.csv`, `det_nh_leaky_rank1.png`–`det_nh_leaky_rank4.png` | Figs. S12, S14, S16, S18; SI text |
| 8. Rank 3 without the lockdown | `no_lockdown_rank3.png` | Fig. S19 |
| 9. The four best fits side by side | `attractor_burden.csv`, `table_S1.csv`, `table_S1.tex` | Table S1 |
| 10. Victoria's re-emergence in the data | (printed) | Materials and Methods |

## Data files (in the same folder as this notebook)
- `FluNet_NH.csv`, `FluNet_Tr.csv`, `FluNet_SH.csv`: WHO FluNet weekly influenza B reports.
- `filtered.csv`: the 595 biologically-filtered parameter samples (Section 3 regenerates it; bundled so that
  Sections 4–10 can run without the search).

## Runtime notes
- `search_1M.csv` (about 200 MB) is **not** bundled. Section 2 generates it; the search is deterministic
  (`seed=42`) but takes about 39 hours on 8 cores. Sections 2–3, Fig. 2 (Section 4) and the summary of the
  selected samples (Section 6) need it; once written, the search is skipped.
- `N_CORES` (Section 0) sets the number of worker processes for the loops over the 595 filtered samples
  (Sections 4 and 7: one model run per sample each). Their results are cached in `filtered_outputs.csv` and
  `leaky_scan.csv`; delete those files to recompute.
- `NSIMS` (Section 0) is the size of each stochastic ensemble: $10^5$ as in the paper, about 40 minutes per
  sample (Section 5 runs four samples). Lower it for a quick qualitative check.
- The figures of Section 5 and 7 save the data they show next to them (`<figure>_plotdata.pkl`);
  `simulator.redraw_fig3('<figure>.png', title=...)` redraws one without rerunning the model.

## 0. Setup

In [ ]:
import os
import contextlib
import datetime
import io
import re
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm.auto import tqdm

import simulator        # the model
import simulator_leaky  # the same model with leaky cross-immunity (SI)

# Computing resources: worker processes for the loops over parameter samples (Sections 2, 4 and 7), and the
# size of each stochastic ensemble (10^5 in the paper; lower it for a quick qualitative check)
N_CORES = min(8, os.cpu_count() or 1)
NSIMS = 100_000

PARAMS = ['R0V', 'R0Y', 'TV_years', 'TY_years', 'sigmaVY', 'sigmaYV', 'm', 'season_phase', 'c_pandemic',
          'Lockdown_duration']
YEAR = 365.25
T_REF_DAYS = 15.75 * YEAR  # model time (days from 1 January 2000) of 1 October 2015, the start of the data
T_REF_DT = datetime.datetime(2015, 10, 1, tzinfo=datetime.timezone.utc)
T_LOCK = 20.25 * YEAR      # 1 April 2020, the start of the lockdown
FIG3 = dict(tspan=27.75 * YEAR, Tpan=20.25 / 27.75)  # the settings of Fig. 3: to July 2027, lockdown on 1 April 2020


def model_kwargs(row):
    """The model's arguments for one parameter sample (a row of the search output)."""
    return dict(R0V=row['R0V'], R0Y=row['R0Y'],
                omegaV=1 / (YEAR * row['TV_years']), omegaY=1 / (YEAR * row['TY_years']),
                sigmaVY=row['sigmaVY'], sigmaYV=row['sigmaYV'], m=row['m'],
                season_phase=row['season_phase'], c_pandemic=row['c_pandemic'],
                Lockdown_duration=row['Lockdown_duration'])


def to_date(t):
    """The calendar date (and time) of model time t, in days."""
    return T_REF_DT.replace(tzinfo=None) + datetime.timedelta(days=float(t) - T_REF_DAYS)


def quiet(fn, *args, **kwargs):
    """fn(*args, **kwargs) without its printed output."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)


# The colours of the SI scatter plots. All samples share one hue, the blue of the filtered samples in the
# violin plots (Fig. S2); the four best fits by NH likelihood are highlighted in every figure, each with its own
# colour, marker and line pattern.
SAMPLES = '#0000ff'
TOP = {  # rank: (colour, marker, line pattern)
    1: ('#5b2a86', 'o', '-'),                       # deep purple
    2: ('#17becf', 's', (0, (4, 1.5))),              # cyan
    3: ('#6b3e1f', '^', (0, (1.2, 1.2))),            # dark brown
    4: ('#f08cc0', 'D', (0, (5, 1.5, 1.2, 1.5))),    # pink
}
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#ffffff'
# the style of the SI scatter plots (Figs. S6-S11), applied with plt.rc_context(STYLE)
STYLE = {
    'font.family': 'Arial', 'font.size': 8, 'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2, 'axes.linewidth': 0.8,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'grid.linestyle': '-', 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'legend.frameon': False, 'legend.fontsize': 7, 'mathtext.default': 'regular',
}


def rank_label(rank):
    return 'Rank 1 (best fit)' if rank == 1 else f'Rank {rank}'


def rank_title(rank, text=''):
    """The title of a figure of one of the four best fits: (text, marker, colour)."""
    colour, marker, _ = TOP[rank]
    return rank_label(rank) + text, marker, colour

## 1. FluNet surveillance data (Fig. 1 and Fig. S1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

df = pd.read_csv('FluNet_NH.csv')

df['ISO_WEEKSTARTDATE'] = pd.to_datetime(df['Week start date (ISO 8601 calendar)'])

start_date = '2015-10-01'
end_date = '2025-09-30'

BYAM = 'B (Yamagata)'
BVIC = 'B (Victoria)'
BNOTDETERMINED = 'B (lineage not determined)'

cols_to_fill = [BYAM, BVIC, BNOTDETERMINED]

filtered_df = df[
        (df['ISO_WEEKSTARTDATE'] >= start_date) &
        (df['ISO_WEEKSTARTDATE'] < end_date)
    ].copy()
    
for col in cols_to_fill:
    filtered_df[col] = filtered_df[col].fillna(0)

# Group by week to aggregate counts
filtered_weekly = filtered_df.groupby('ISO_WEEKSTARTDATE')[cols_to_fill].sum().reset_index()

# Calculate total B
filtered_weekly['Total_B'] = filtered_weekly[BYAM] + filtered_weekly[BVIC]

# Filter out weeks with 0 cases
plot_data = filtered_weekly[filtered_weekly['Total_B'] > 0].copy()
    

# Pandemic cutoff date
pandemic_cutoff = pd.to_datetime('2020-04-01')
testing_cutoff = pd.to_datetime('2020-07-01')

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(plot_data['ISO_WEEKSTARTDATE'], np.sqrt(plot_data[BVIC]),  label='Victoria')
ax.plot(plot_data['ISO_WEEKSTARTDATE'], np.sqrt(plot_data[BYAM]),  label='Yamagata')
ax.plot(plot_data['ISO_WEEKSTARTDATE'], np.sqrt(plot_data[BNOTDETERMINED]), label='Untyped')
ax.axvline(pandemic_cutoff, color='red', linestyle='--', linewidth=2, alpha=0.7)

lim = np.sqrt(1.05 * max(plot_data[[BVIC, BYAM, BNOTDETERMINED]].max().max(), 1))
ax.text(pandemic_cutoff, lim*0.95, 'lockdown', color='red', fontsize=14,  alpha=0.7, va='top', ha='left', rotation=90)
# Render July 1st 2020 Testing Increase Line
#ax.axvline(testing_cutoff, color='green', linestyle='-.', linewidth=2, alpha=0.3)
ax.set_ylim(0, lim)
ax.set_ylabel('weekly reports', fontsize=14)
#ax.set_title('Northern Hemisphere - Weekly B lineage reports')
ax.legend(loc='upper left', fontsize=14)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: f'{int(val**2):,}'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax.xaxis.get_majorticklabels(), fontsize=12)
plt.setp(ax.yaxis.get_majorticklabels(), fontsize=12)
# Set x-axis limits
ax.set_xlim(pd.Timestamp(start_date), pd.Timestamp(end_date))

plt.tight_layout()
plt.savefig('nh_data.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# Load the data for all three regions
df_nh = pd.read_csv('FluNet_NH.csv')
df_tr = pd.read_csv('FluNet_Tr.csv')
df_sh = pd.read_csv('FluNet_SH.csv')

dfs = [df_nh, df_tr, df_sh]
# Convert date column to datetime objects
for df in dfs:
    df['ISO_WEEKSTARTDATE'] = pd.to_datetime(df['Week start date (ISO 8601 calendar)'])

start_date ='2015-10-01'
end_date = '2025-09-30'

BYAM = 'B (Yamagata)'
BVIC = 'B (Victoria)'
BNOTDETERMINED = 'B (lineage not determined)'

cols_to_fill = [BYAM, BVIC, BNOTDETERMINED]
region_names = ["Northern Hemisphere", "Tropics", "Southern Hemisphere"]
region_dfs = []

# Process each region
for region_df in dfs:
    filtered_df = region_df[
        (region_df['ISO_WEEKSTARTDATE'] >= start_date) &
        (region_df['ISO_WEEKSTARTDATE'] < end_date)
    ].copy()
    
    for col in cols_to_fill:
        filtered_df[col] = filtered_df[col].fillna(0)
    
    # Group by week to aggregate counts
    filtered_weekly = filtered_df.groupby('ISO_WEEKSTARTDATE')[cols_to_fill].sum().reset_index()
    
    # Calculate total B
    filtered_weekly['Total_B'] = filtered_weekly[BYAM] + filtered_weekly[BVIC]
    
    # Filter out weeks with 0 cases
    plot_data = filtered_weekly[filtered_weekly['Total_B'] > 0].copy()
    
    region_dfs.append(plot_data)
    

# Pandemic cutoff date
pandemic_cutoff = pd.to_datetime('2020-04-01')
testing_cutoff = pd.to_datetime('2020-07-01')

# Create subplots for all three regions
fig, axes = plt.subplots(len(dfs), 1, figsize=(15, 12), sharex=True)

for idx, (ax, region_data) in enumerate(zip(axes, region_dfs)):
    ax.plot(region_data['ISO_WEEKSTARTDATE'], np.sqrt(region_data[BVIC]), label='Victoria')
    ax.plot(region_data['ISO_WEEKSTARTDATE'], np.sqrt(region_data[BYAM]), label='Yamagata')
    ax.plot(region_data['ISO_WEEKSTARTDATE'], np.sqrt(region_data[BNOTDETERMINED]), label='Untyped')
    lim=np.sqrt(1.05*max(region_data[BVIC].max(), region_data[BYAM].max(), region_data[BNOTDETERMINED].max()))
    ax.vlines(pandemic_cutoff, 0, lim, 
            colors='red', linestyles='--', linewidth=2, alpha=0.7)
    ax.text(pandemic_cutoff, lim*0.95, 'lockdown', color='red', fontsize=14,  alpha=0.7, va='top', ha='left', rotation=90)
    ax.set_ylim(0, lim)
    ax.set_ylabel('weekly reports', fontsize=14)
    ax.set_title(f'{region_names[idx]}', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: f'{int(val**2):,}'))
    # Format x-axis for bottom subplot
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.setp(axes[-1].xaxis.get_majorticklabels())


# Set x-axis limits
for ax in axes:
    ax.set_xlim(pd.Timestamp(start_date), pd.Timestamp(end_date))
    plt.setp(ax.yaxis.get_majorticklabels(), fontsize=12)
#axes[-1].set_xlabel('ISO Week Start Date', fontsize=11)
plt.setp(axes[-1].xaxis.get_majorticklabels(), fontsize=12)

plt.tight_layout()
plt.savefig('data.png', dpi=300)
plt.show()

## 2. Latin hypercube parameter search (produces `search_1M.csv`)

The cell below defines the parallel Latin hypercube sampling (LHS) search over the ten model parameters.
`search_1M.csv` is not bundled with this deposit, so the search must be run once to generate it. The search
is **deterministic** (`seed=42`) but expensive (about 39 hours on 8 cores); once the file exists, it is reused
and the search is skipped. Each row of `search_1M.csv` is one successful model run; its row number identifies
the sample in every later file (the index of `target_runs.csv`, `filtered.csv` and `filtered_outputs.csv`).

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import qmc
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm # tqdm.auto formats beautifully in Jupyter

# 1. Import the wrapper from your file! 
from simulator import run_single_simulation
#WITH THRESHOLD one FOR EXTINCTION

# 2. Define your bounds
param_names = ['R0V', 'R0Y', 'TV', 'TY', 'sigmaVY', 'sigmaYV', 'm', 'season_phase', 'c_pandemic', 'Lockdown_duration']
lower_bounds = [1, 1, 1, 1, 0.0, 0.0, 0.0010,0,0.2,100]
upper_bounds = [5, 5, 10,  10,  1.0, 1.0, 0.01, 100, 0.8,600] 

def run_parallel_lhs_jupyter(n_samples=2000, n_cores=8):
    print(f"Setting up Latin Hypercube with {n_samples} samples...")
    sampler = qmc.LatinHypercube(d=len(param_names), seed=42)
    sample_scaled = qmc.scale(sampler.random(n=n_samples), lower_bounds, upper_bounds)
    
    tasks = [row for row in sample_scaled]
    results = []
    failed =[]
    
   
    
    if n_cores == 1:
        print("Running simulations sequentially...")
        for task in tqdm(tasks, total=n_samples):
            res = run_single_simulation(task)
            if res['success']:
                results.append(res)
            else:
                failed.append(res)
    else:
        # ProcessPoolExecutor handles the core management
        with ProcessPoolExecutor(max_workers=n_cores) as executor:
            print(f"Firing up {n_cores} cores...")
            # Map the imported wrapper function to your LHS tasks
            #for res in tqdm(executor.map(run_single_simulation, tasks), total=n_samples)
            for res in tqdm(executor.map(run_single_simulation, sample_scaled, chunksize=100), total=n_samples):
                (results if res['success'] else failed).append(res)
                
    # Clean up and filter
    df = pd.DataFrame(results).drop(columns=['success'])
    
    print("-" * 50)
    print(f"Total successful ODE runs: {len(df)} / {n_samples}")
    print("-" * 50)
    
    return df, failed

In [ ]:
# Runs the one-million-sample search and writes search_1M.csv, unless it is already there
if not os.path.exists('search_1M.csv'):
    all_res, failed = run_parallel_lhs_jupyter(n_samples=1_000_000, n_cores=N_CORES)
    all_res.to_csv('search_1M.csv', index=False)
    print(f'Wrote search_1M.csv with {len(all_res):,} successful runs.')
all_res = pd.read_csv('search_1M.csv')
print(f'Loaded {len(all_res):,} parameter samples from search_1M.csv')

## 3. Selection, biological filter and parameter space (Table 1, Figs. S2–S5)

The selection rules (`count_and_plot`): coexistence before 2020, Yamagata extinct after the lockdown and
before Victoria's re-emergence, Victoria surviving and re-emerging 1.5–2 years after the lockdown (October
2021 to March 2022). The selected samples are written to `target_runs.csv`; those that also satisfy the
biological filter ($T_V > T_Y$ and $\sigma_{VY} > \sigma_{YV}$) to `filtered.csv`. Then the violin plots
(Fig. S2), the pairwise densities (Figs. S4–S5), the XGBoost + SHAP feature importance (Fig. S3) and Table 1.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.ticker import MaxNLocator
from scipy.stats import gaussian_kde

# Latin-hypercube bounds (Section 2): parameters are normalised to, and plotted over, these ranges
LHS_BOUNDS = {'R0V': (1, 5), 'R0Y': (1, 5), 'TV_years': (1, 10), 'TY_years': (1, 10), 'sigmaVY': (0, 1),
              'sigmaYV': (0, 1), 'm': (0.001, 0.01), 'season_phase': (0, 100), 'c_pandemic': (0.2, 0.8),
              'Lockdown_duration': (100, 600)}

SHORT_LABELS = {'R0V': r"$R_{0,V}$", 'R0Y': r"$R_{0,Y}$", 'TV_years': r"$T_{V}$", 'TY_years': r"$T_{Y}$",
                'sigmaVY': r"$\sigma_{VY}$", 'sigmaYV': r"$\sigma_{YV}$", 'm': r"$m$", 'season_phase': r"$\phi$",
                'c_pandemic': r"$c_0$", 'Lockdown_duration': r"$\tau$"}


def plot_pairwise_density(df, colour, cmap, name=None):
    """
    Pairwise marginal densities of the ten parameters (Figs. S4-S5):
    univariate densities on the diagonal, bivariate densities below it, all over the
    Latin-hypercube bounds. Drawn at its printed size (the SI text width).
    """
    params = list(LHS_BOUNDS)
    n = len(params)
    fig, axes = plt.subplots(n, n, figsize=(7.0, 7.0))
    grid = {p: np.linspace(*LHS_BOUNDS[p], 60) for p in params}
    for i, py in enumerate(params):
        for j, px in enumerate(params):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
                continue
            if i == j:
                xs = np.linspace(*LHS_BOUNDS[px], 200)
                dens = gaussian_kde(df[px].values)(xs)
                ax.fill_between(xs, dens, color=colour, alpha=0.35, linewidth=0)
                ax.plot(xs, dens, color=colour, linewidth=0.8)
                ax.set_ylim(0, 1.1 * dens.max())
            else:
                X, Y = np.meshgrid(grid[px], grid[py])
                kde = gaussian_kde(np.vstack([df[px].values, df[py].values]))
                Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
                levels = np.linspace(0.05 * Z.max(), Z.max(), 8)  # lowest 5% left blank
                ax.contourf(X, Y, Z, levels=levels, cmap=cmap)
                ax.set_ylim(LHS_BOUNDS[py])
            ax.set_xlim(LHS_BOUNDS[px])
            ax.xaxis.set_major_locator(MaxNLocator(3, prune='both'))
            ax.yaxis.set_major_locator(MaxNLocator(3, prune='both'))
            ax.tick_params(labelsize=5.5, length=2, pad=1)
            if i == j:
                ax.set_yticks([])  # density scale not shown on the diagonal
            if i == n - 1:
                ax.set_xlabel(SHORT_LABELS[px], fontsize=8, labelpad=1)
            else:
                ax.set_xticklabels([])
            if j == 0 and i > 0:
                ax.set_ylabel(SHORT_LABELS[py], fontsize=8, labelpad=1)
            elif i != j:
                ax.set_yticklabels([])
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.07, top=0.99, wspace=0.12, hspace=0.12)
    if name is not None:
        plt.savefig(name, dpi=300)
    plt.show()


def plot_parameter_space(df, name=None):
    """
    Takes the full DataFrame from the LHS sweep and plots the parameter spaces.
    Duplicates the 'brown' subset rows so they appear in both the full 'green'
    distribution and their own 'brown' distribution.
    The violin plot (Fig. S2) and the pairwise densities (Figs. S4-S5) are drawn at their printed
    size (the SI text width), so the font sizes below are the printed sizes.
    """
    plot_df = df.copy()

    coexistence_target = (df['simple_coexist'] == True) & (df['recovery_Y'] !=-1) & (df['recovery_V'] != -1)
    Yextinction_Vrecovery = coexistence_target & (df['extinct_Y'] == True) & (df['extinct_V'] == False) & (df['recovery_Y'] != -100)

    str1 = 'Vic and Yam coexist before COVID-19'
    str3 = 'Yam extinct, Vic reemerges Oct 2021 - Apr 2022 (All)'
    str_blue = 'Yam extinct, Vic reemerges Oct 2021 - Apr 2022 (Favored Subset)'

    # ==========================================
    # DEFINING OUTCOMES (Overlapping Logic)
    # ==========================================

    # 1. Define the base Green condition
    cond_green = Yextinction_Vrecovery & (df['recovery_V'] > 1.5) & (df['recovery_V'] <= 2)

    # 2. Assign the baseline and the FULL green group (this includes the brown ones for now)
    plot_df['Outcome'] = np.select([cond_green], [str3], default=str1)

    # 3. Define the B subset condition
    cond_blue = cond_green & (df['sigmaVY'] > df['sigmaYV'])  & (df['TV_years'] > df['TY_years']) #& (df['R0V'] > df['R0Y'])

    # 4. DUPLICATE the blue rows so they appear in both categories
    brown_df = plot_df[cond_blue].copy()
    brown_df['Outcome'] = str_blue # Relabel the duplicates

    # Concatenate them back together
    plot_df = pd.concat([plot_df, brown_df], ignore_index=True)

    # Define the exact columns from the dataframe
    param_cols = ['R0V', 'R0Y', 'TV_years', 'TY_years', 'sigmaVY', 'sigmaYV', 'm', 'season_phase', 'c_pandemic', 'Lockdown_duration']

    # Create a dictionary mapping the columns to their display names
    display_labels = {'R0V': "B/Vic's basic reproduction number: " + r"$R_{0,V}$",
        'R0Y': "B/Yam's basic reproduction number: " + r"$R_{0,Y}$",
        'TV_years': "Average duration of B/Vic's immunity (years): " + r"$T_{V}$",
        'TY_years': "Average duration of B/Yam's immunity (years): " + r"$T_{Y}$",
        'sigmaVY': "Cross-immunity from B/Vic to B/Yam: " + r"$\sigma_{VY}$",
        'sigmaYV': "Cross-immunity from B/Yam to B/Vic: " + r"$\sigma_{YV}$",
        'm': "Inter-region contact rate (relative): " + r"$m$",
        'season_phase': "Peak seasonality in NH (days since Jan 1st): " + r"$\phi$",
        'c_pandemic': "Initial COVID-19 reduction in contacts (relative): "+r"$c_0$",
        'Lockdown_duration': 'Lockdown timescale (days): ' + r"$\tau$",
    }

    # ==========================================
    # DEFINING COLORS
    # ==========================================
    colors = ['lightgrey', 'green', 'blue']
    palette = {
        str1: colors[0],
        str3: colors[1],
        str_blue: colors[2]
    }

    # ==========================================
    # PLOT: Violin Plots (Fig. S2)
    # ==========================================
    print("Generating Violin Plots...")

    min_max_dict = {col: LHS_BOUNDS[col] for col in param_cols}

    # Normalize the data to the Latin-hypercube bounds
    norm_df = plot_df.copy()
    for col in param_cols:
        min_val, max_val = min_max_dict[col]
        norm_df[col] = (norm_df[col] - min_val) / (max_val - min_val)

    # Melt the dataframe
    melted_df = norm_df.melt(
        id_vars=['Outcome'],
        value_vars=param_cols,
        var_name='Parameter',
        value_name='Normalized Value'
    )

    # Swap the original column names for the pretty labels, over two lines so they fit at print size
    violin_labels = {
        'R0V': "B/Vic's basic reproduction\nnumber: " + r"$R_{0,V}$",
        'R0Y': "B/Yam's basic reproduction\nnumber: " + r"$R_{0,Y}$",
        'TV_years': "Average duration of B/Vic's\nimmunity (years): " + r"$T_{V}$",
        'TY_years': "Average duration of B/Yam's\nimmunity (years): " + r"$T_{Y}$",
        'sigmaVY': "Cross-immunity from\nB/Vic to B/Yam: " + r"$\sigma_{VY}$",
        'sigmaYV': "Cross-immunity from\nB/Yam to B/Vic: " + r"$\sigma_{YV}$",
        'm': "Inter-region contact\nrate (relative): " + r"$m$",
        'season_phase': "Peak seasonality in NH\n(days since Jan 1st): " + r"$\phi$",
        'c_pandemic': "Initial COVID-19 reduction\nin contacts (relative): " + r"$c_0$",
        'Lockdown_duration': "Lockdown timescale\n(days): " + r"$\tau$",
    }
    melted_df['Parameter'] = melted_df['Parameter'].map(violin_labels).fillna(melted_df['Parameter'])

    fig, ax = plt.subplots(figsize=(7.0, 7.4))

    # Explicitly define the hue order to calculate text spacing offsets reliably
    hue_order_list = [str1, str3, str_blue]

    sns.violinplot(
        data=melted_df,
        y='Parameter',
        x='Normalized Value',
        hue='Outcome',
        hue_order=hue_order_list,
        palette=palette,
        inner='quartile',
        linewidth=0.6,
        cut=0,
        ax=ax
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

    ax.set_xticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xlim(0, 1)

    ax.tick_params(axis='y', pad=28, length=0, labelsize=7.5)

    # Define vertical dodge offsets for 3 categories in Seaborn
    hue_offsets = [-0.267, 0, 0.267]

    # Summary statistics go in a column to the right of the violins rather than on top of them
    #: median (mean) [25th-75th percentile], one line per group
    ax.text(1.225, -0.75, 'Median (mean) [IQR]', ha='left', va='center', fontsize=7, fontweight='bold',
            clip_on=False)
    for i, col in enumerate(param_cols):
        min_val, max_val = min_max_dict[col]

        # Custom display formatting logic per column rules
        if col == 'Lockdown_duration' or col =='season_phase':
            fmt = lambda x: f"{x:.0f}"
        elif col == 'm':
            fmt = lambda x: f"{x:.4f}"
        else:
            fmt = lambda x: f"{x:.2f}"

        # Sampling bounds at either end of the violins, written as short as possible (e.g. 0.001, 10)
        ax.text(-0.02, i, f"{min_val:g}", va='center', ha='right', fontsize=7, color='black', clip_on=False)
        ax.text(1.02, i, f"{max_val:g}", va='center', ha='left', fontsize=7, color='black', clip_on=False)

        for j, outcome in enumerate(hue_order_list):
            data_raw = plot_df[plot_df['Outcome'] == outcome][col].dropna().values
            if len(data_raw) < 2:
                continue
            q25, q50, q75 = np.percentile(data_raw, [25, 50, 75])
            mean = np.mean(data_raw)
            y_pos = i + hue_offsets[j]
            ax.plot(1.19, y_pos, 's', color=palette[outcome], markersize=4, markeredgecolor='dimgray',
                    markeredgewidth=0.3, clip_on=False)
            ax.text(1.225, y_pos, f'{fmt(q50)} ({fmt(mean)}) [{fmt(q25)}–{fmt(q75)}]', ha='left', va='center',
                    fontsize=6.5, color='black', clip_on=False)

    legend_handles = [mlines.Line2D([], [], color=c, marker='s', linestyle='None', markersize=6,
                                    markeredgecolor='dimgray', markeredgewidth=0.3, label=l)
                      for c, l in zip(colors, ['Coexistence before COVID-19',
                                               'Selected: Yam extinct, Vic reemerges Oct 2021 – Mar 2022',
                                               r'Filtered: selected, with $T_V > T_Y$ and $\sigma_{VY} > \sigma_{YV}$'])]
    fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 1.0), ncol=1, fontsize=7.5,
               frameon=False, handletextpad=0.3, columnspacing=1.0)

    sns.despine(ax=ax, left=True, bottom=True)
    fig.subplots_adjust(left=0.30, right=0.58, top=0.905, bottom=0.01)

    if name is not None:
        plt.savefig(name, dpi=300)
    plt.show()

    # ==========================================
    # PLOT 3: Pairwise densities (Figs. S4-S5)
    # ==========================================
    print("Generating Pairwise Density Plots...")
    for subset, colour, cmap, suffix in [(cond_green, 'green', 'Greens', '_pairs_green'),
                                         (cond_blue, 'blue', 'Blues', '_pairs_blue')]:
        save_name = None
        if name is not None:
            save_name = name.replace(".png", f"{suffix}.png") if ".png" in name else name + suffix
        plot_pairwise_density(df[subset], colour, cmap, save_name)

In [ ]:
def count_and_plot(df, name=None):
    L=len(df)
    print('Number of successful ODE runs:', L)
    df=df[(df['recovery_Y'] !=-1) & (df['recovery_V'] != -1) & (df['simple_coexist'] == True) ]
    input_df = df
    print('Percentage of successful runs with pre-pandemic coexistence:', len(df)/L*100)
    df = df[(df['extinct_Y'] == True) & (df['extinct_V'] == False) & (df['recovery_Y'] != -100)]
    print('Percentage of successful runs with Y extinct and V surviving:', len(df)/L*100)
    print('Percentage of successful runs with early V recovery:', len(df[df['recovery_V'] <= 1.5])/L*100)
    target_runs = df[(df['recovery_V'] > 1.5) & (df['recovery_V'] <= 2)]
    print('Percentage of successful runs with appropriate V recovery:', len(target_runs)/L*100)
    print('Percentage of successful runs with late V recovery:', len(df[df['recovery_V'] > 2])/L*100)
    plot_parameter_space(input_df, name=name)
    return target_runs

In [ ]:
# Fig. S2 (violin plots) and Figs. S4-S5 (pairwise densities of the selected and of the filtered samples)
target_runs = count_and_plot(all_res, 'both_outcomes.png')
target_runs.to_csv('target_runs.csv')  # the selected samples; index: row in search_1M.csv

# the biological filter
filtered = target_runs[(target_runs['sigmaVY'] > target_runs['sigmaYV']) & (target_runs['TV_years'] > target_runs['TY_years'])]
filtered.to_csv('filtered.csv')
print(f'Selected samples: {len(target_runs):,}; filtered: {len(filtered):,} '
      f'({100 * len(filtered) / len(target_runs):.0f}% of the selected, {100 * len(filtered) / 1e6:.2f}% of all samples)')

In [ ]:
import xgboost as xgb
import shap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# â”€â”€ Parameters â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
param_cols = ['R0V', 'R0Y', 'TV_years', 'TY_years', 'sigmaVY', 'sigmaYV',
              'm', 'season_phase', 'c_pandemic', 'Lockdown_duration']



NEG_TO_POS_RATIO = 10   # keep this many negatives per positive after undersampling
RANDOM_STATE     = 42

# â”€â”€ Binary outcome (same definition as cond_green used elsewhere) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_df = all_res.copy()
#_df = _df[(_df['recovery_Y'] != -1) & (_df['recovery_V'] != -1) & (_df['simple_coexist'] == True)]

coexistence_target = (_df['simple_coexist'] == True) & (_df['recovery_Y'] != -1) & (_df['recovery_V'] != -1)
Yextinction_Vrecovery = coexistence_target & (_df['extinct_Y'] == True) & (_df['extinct_V'] == False) & (_df['recovery_Y'] != -100)

y_all = (Yextinction_Vrecovery & (_df['recovery_V'] > 1.5) & (_df['recovery_V'] <= 2)).astype(int)

X_all = _df[param_cols].reset_index(drop=True)
y_all = y_all.reset_index(drop=True)

n_pos = int(y_all.sum())
n_neg = int((y_all == 0).sum())
print(f"Class balance: {n_pos:,} pos / {n_neg:,} neg  ({n_pos/(n_pos+n_neg)*100:.3f}% positive)")

# â”€â”€ Stratified undersample â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
pos_idx  = y_all[y_all == 1].index
neg_idx  = y_all[y_all == 0].index
n_keep   = min(len(neg_idx), len(pos_idx) * NEG_TO_POS_RATIO)
neg_keep = neg_idx.to_series().sample(n=n_keep, random_state=RANDOM_STATE).index
keep     = pos_idx.union(neg_keep)

X_bal = X_all.loc[keep]
y_bal = y_all.loc[keep]
print(f"Downsampled: {int(y_bal.sum()):,} pos / {int((y_bal==0).sum()):,} neg  ({NEG_TO_POS_RATIO}:1 ratio)")

# â”€â”€ XGBoost â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
model.fit(X_bal, y_bal)
print("XGBoost training done.")

# â”€â”€ SHAP â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
explainer = shap.TreeExplainer(model)
shap_raw  = explainer.shap_values(X_bal)
shap_vals = shap_raw[1] if isinstance(shap_raw, list) else shap_raw   # positive class

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=param_cols, name="Mean |SHAP|")
display_labels = {
    'R0V':              r"$R_{0,V}$",
    'R0Y':              r"$R_{0,Y}$",
    'TV_years':         r"$T_V$",
    'TY_years':         r"$T_Y$",
    'sigmaVY':          r"$\sigma_{VY}$",
    'sigmaYV':          r"$\sigma_{YV}$",
    'm':                r"$m$",
    'season_phase':     r"$\phi$",
    'c_pandemic':       r"$c_0$",
    'Lockdown_duration': r"$\tau$",
}
X_bal_labeled = X_bal.rename(columns=display_labels)

# mapping: display label -> mean |SHAP|
label_to_shap = {display_labels.get(col, col): mean_abs_shap[i]
                 for i, col in enumerate(param_cols)}

# -- Plot 1: beeswarm (direction + spread) ------------------------------------
# the points are shuffled and jittered at random: seeded so that the figure is the same on every run
shap.summary_plot(shap_vals, X_bal_labeled, max_display=len(param_cols), show=False,
                  rng=np.random.default_rng(RANDOM_STATE))
ax = plt.gca()
ax.set_yticklabels([
    f"{lbl.get_text()}  ({label_to_shap[lbl.get_text()]:.1f})"
    if lbl.get_text() in label_to_shap else lbl.get_text()
    for lbl in ax.get_yticklabels()
])
ax.text(0.135, 0.95, "Feature (mean |SHAP|)",
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=12, fontweight='bold', clip_on=False)
#plt.title('SHAP Beeswarm', fontsize=13, weight='bold', pad=10)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()


print("\nMean |SHAP| (descending):")
print(shap_importance.sort_values(ascending=False).to_string())

In [ ]:
# Table 1: the distribution of each parameter among the selected samples (mean; 25th, 50th and 75th
# percentiles) and its mean |SHAP| value
table_1 = pd.DataFrame({
    'lower bound': [LHS_BOUNDS[p][0] for p in PARAMS], 'upper bound': [LHS_BOUNDS[p][1] for p in PARAMS],
    'mean': target_runs[PARAMS].mean().values,
    'p25': target_runs[PARAMS].quantile(0.25).values, 'median': target_runs[PARAMS].median().values,
    'p75': target_runs[PARAMS].quantile(0.75).values,
    'mean |SHAP|': shap_importance[PARAMS].values}, index=PARAMS)
table_1.to_csv('table_1.csv')
table_1.round(4)

## 4. Likelihood of the filtered samples (Fig. 2, Figs. S10, S11, S20–S23)

The Poisson likelihood of the Northern-Hemisphere (NH) FluNet data, with the reporting rates $\rho_-$
(until 30 June 2020) and $\rho_+$ (from 1 July 2020) profiled out, for each of the 595 filtered samples. The
same deterministic run of each sample (Fig. 3 settings) gives every other per-sample output used below:
Victoria's minimum global prevalence after the lockdown and its date, Yamagata's extinction date, and
Yamagata's effective reproduction number $R_e$ in the NH at the peak of the 2026–27 season and on the four
hypothetical reintroduction dates. They are saved in `filtered_outputs.csv`, ranked by NH likelihood (rank 1
is the best fit).

In [ ]:
import matplotlib.dates as mdates

# The weekly reports of each region, 1 October 2015 to 30 September 2025, without the weeks with no B reports
BYAM = 'B (Yamagata)'
BVIC = 'B (Victoria)'
BNOTDETERMINED = 'B (lineage not determined)'
start_date, end_date = '2015-10-01', '2025-09-30'
obs_dfs_list = []
for region_file in ['FluNet_NH.csv', 'FluNet_Tr.csv', 'FluNet_SH.csv']:
    region_df = pd.read_csv(region_file)
    region_df['ISO_WEEKSTARTDATE'] = pd.to_datetime(region_df['Week start date (ISO 8601 calendar)'])
    filtered_df = region_df[(region_df['ISO_WEEKSTARTDATE'] >= start_date) &
                            (region_df['ISO_WEEKSTARTDATE'] < end_date)].copy()
    for col in [BYAM, BVIC, BNOTDETERMINED]:
        filtered_df[col] = filtered_df[col].fillna(0)
    filtered_weekly = filtered_df.groupby('ISO_WEEKSTARTDATE')[[BYAM, BVIC, BNOTDETERMINED]].sum().reset_index()
    filtered_weekly = filtered_weekly[
        (filtered_weekly[BYAM] + filtered_weekly[BVIC] + filtered_weekly[BNOTDETERMINED]) > 0].copy()
    obs_dfs_list.append(filtered_weekly)

t_ref_days, t_ref_dt = T_REF_DAYS, T_REF_DT


def compute_nll_from_result(result, obs_dfs_list, verbose=False):
    """
    Negative log-likelihood of a model run (result of simulator.model) for each of the 3 regions, with
    different reporting rates before and after 1 July 2020 (profiled out analytically). Returns the total
    NLL, the reporting rates (rho_pre, rho_post) of each region, the matched predictions and observations,
    and diagnostics (diagnostics['nll_by_region'][0] is the NH NLL).
    """
    weekly_times = result['weekly_times']
    
    # Convert simulator times to ISO week dates
    def t_to_iso_date(t_days):
        """Convert simulation time (days) to ISO week date string."""
        days_from_ref = t_days - t_ref_days
        target_date = t_ref_dt + datetime.timedelta(days=days_from_ref)
        iso_cal = target_date.isocalendar()
        return f"{iso_cal[0]}-W{iso_cal[1]:02d}"
    
    sim_iso_dates = [t_to_iso_date(t) for t in weekly_times]
    
    # Pandemic cutoff date
    testing_cutoff = pd.to_datetime('2020-07-01')
    
    nll_total = 0.0
    rho_dict = {}
    predictions_dict = {}
    diagnostics = {'nll_by_region': {}}
    
    eps = 1e-10
    
    # Evaluate likelihood for each region
    for region_idx, obs_df in enumerate(obs_dfs_list):
        # Extract weekly predictions for this region
        weekly_V = result['weekly_infections_V'][region_idx]
        weekly_Y = result['weekly_infections_Y'][region_idx]
        
        # Match simulator weeks to observation weeks
        obs_iso_dates = obs_df['ISO_WEEKSTARTDATE'].dt.strftime('%G-W%V').values
        
        muV_matched = []
        muY_matched = []
        
        for obs_iso in obs_iso_dates:
            if obs_iso in sim_iso_dates:
                idx = sim_iso_dates.index(obs_iso)
                muV_matched.append(weekly_V[idx])
                muY_matched.append(weekly_Y[idx])
            else:
                muV_matched.append(eps)
                muY_matched.append(eps)
        
        muV = np.array(muV_matched)
        muY = np.array(muY_matched)
        
        muV = np.maximum(muV, eps)
        muY = np.maximum(muY, eps)
        muT = np.maximum(muV + muY, eps)
        
        # Weekly typing probability from data
        obsV = obs_df[BVIC].values
        obsY = obs_df[BYAM].values
        obsN = obs_df[BNOTDETERMINED].values
        
        typed = obsV + obsY
        total = typed + obsN
        tau_t = np.where(total > 0, typed / total, 0.5)
        tau_t = np.clip(tau_t, eps, 1.0 - eps)
        
        # Separate pre- and post-pandemic data
        obs_dates = obs_df['ISO_WEEKSTARTDATE'].values
        pre_pandemic_mask = obs_dates < testing_cutoff
        post_pandemic_mask = obs_dates >= testing_cutoff
        
        # Profile out rho analytically for PRE-pandemic period
        sum_mu_total_pre = float(np.sum(muT[pre_pandemic_mask]))
        sum_obs_total_pre = float(np.sum(obsV[pre_pandemic_mask]) + np.sum(obsY[pre_pandemic_mask]) + np.sum(obsN[pre_pandemic_mask]))
        
        # Profile out rho analytically for POST-pandemic period
        sum_mu_total_post = float(np.sum(muT[post_pandemic_mask]))
        sum_obs_total_post = float(np.sum(obsV[post_pandemic_mask]) + np.sum(obsY[post_pandemic_mask]) + np.sum(obsN[post_pandemic_mask]))
        
        if sum_mu_total_pre <= 0 or sum_mu_total_post <= 0:
            return np.inf, None, None, {'error': f'sum_mu_total <= 0 for region {region_idx}', 'success': False}
        
        rho_pre = sum_obs_total_pre / sum_mu_total_pre if sum_obs_total_pre > 0 else eps
        rho_post = sum_obs_total_post / sum_mu_total_post if sum_obs_total_post > 0 else eps
        
        rho_dict[region_idx] = (rho_pre, rho_post)
        predictions_dict[region_idx] = {'muV': muV, 'muY': muY, 'obsV': obsV, 'obsY': obsY, 'obsN': obsN, 'tau_t': tau_t}
        
        # Compute negative log-likelihood for PRE-pandemic period
        ll_core_pre = (
            np.sum(obsV[pre_pandemic_mask] * (np.log(tau_t[pre_pandemic_mask]) + np.log(muV[pre_pandemic_mask])))
            + np.sum(obsY[pre_pandemic_mask] * (np.log(tau_t[pre_pandemic_mask]) + np.log(muY[pre_pandemic_mask])))
            + np.sum(obsN[pre_pandemic_mask] * (np.log(1.0 - tau_t[pre_pandemic_mask]) + np.log(muT[pre_pandemic_mask])))
        )
        
        nll_pre = sum_obs_total_pre * np.log(sum_mu_total_pre) - ll_core_pre
        
        # Compute negative log-likelihood for POST-pandemic period
        ll_core_post = (
            np.sum(obsV[post_pandemic_mask] * (np.log(tau_t[post_pandemic_mask]) + np.log(muV[post_pandemic_mask])))
            + np.sum(obsY[post_pandemic_mask] * (np.log(tau_t[post_pandemic_mask]) + np.log(muY[post_pandemic_mask])))
            + np.sum(obsN[post_pandemic_mask] * (np.log(1.0 - tau_t[post_pandemic_mask]) + np.log(muT[post_pandemic_mask])))
        )
        
        nll_post = sum_obs_total_post * np.log(sum_mu_total_post) - ll_core_post
        nll_region = nll_pre + nll_post
        
        # Store the individual region's NLL
        diagnostics['nll_by_region'][region_idx] = nll_region
        
        nll_total += nll_region
    
    diagnostics['success'] = True
    
    if verbose:
        print(f"Total NLL (all regions): {nll_total:.2f}")
    
    return float(nll_total), rho_dict, predictions_dict, diagnostics


def compute_nll_for_params_all_regions(row, obs_dfs_list, verbose=False):
    """compute_nll_from_result for one parameter sample, run with the Fig. 3 settings."""
    result = simulator.model(plot=False, stochastic=False, **model_kwargs(row))
    return compute_nll_from_result(result, obs_dfs_list, verbose)

In [ ]:
# One deterministic run per filtered sample (Fig. 3 settings; nsims2=0 gives Yamagata's R_e on the four
# reintroduction dates without the stochastic ensembles), in parallel. Cached in filtered_outputs.csv.
filtered = pd.read_csv('filtered.csv', index_col=0)  # the 595 filtered samples; index: row in search_1M.csv
RE_COLUMNS = ['Re_Oct1', 'Re_Jan1', 'Re_Apr1', 'Re_Jul1']  # 1 October 2026, 1 January, 1 April, 1 July 2027


def sample_outputs(res):
    nll, rho_dict, _, diag = compute_nll_from_result(res, obs_dfs_list)
    return {'NLL_nh': diag['nll_by_region'][0], 'rho_pre': rho_dict[0][0], 'rho_post': rho_dict[0][1],
            'min_prevalence_V': res['min_prevalence_V'], 't_min_prevalence_V': res['t_min_prevalence_V'],
            't_yamagata_extinction': res['t_yamagata_extinction'], 'recovery_V': res['recovery_V'],
            'R_eff_Y_peak_2026_27': float(np.ravel(res['R_eff_Y_peak_2026_27'])[0]),
            't_R_eff_Y_peak_2026_27': res['t_R_eff_Y_peak_2026_27'], **dict(zip(RE_COLUMNS, res['R_values']))}


if os.path.exists('filtered_outputs.csv'):
    outputs = pd.read_csv('filtered_outputs.csv', index_col=0)
else:
    rows = {}
    with ProcessPoolExecutor(max_workers=N_CORES) as pool:
        pending = {pool.submit(simulator.model, plot=False, stochastic=False, stochastic2=True, nsims2=0,
                               **FIG3, **model_kwargs(row)): idx for idx, row in filtered.iterrows()}
        for future in tqdm(as_completed(pending), total=len(pending)):
            rows[pending.pop(future)] = sample_outputs(future.result())
    outputs = filtered[PARAMS].join(pd.DataFrame.from_dict(rows, orient='index')).sort_values('NLL_nh')
    outputs.insert(0, 'rank', np.arange(1, len(outputs) + 1))
    outputs.index.name = 'sample'
    outputs.to_csv('filtered_outputs.csv')

RANKED = outputs.sort_values('rank').index.tolist()  # the samples by NH likelihood, best first
outputs.head(4)[['rank', 'NLL_nh', 'rho_pre', 'rho_post', 'min_prevalence_V', 'R_eff_Y_peak_2026_27'] + RE_COLUMNS]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Latin-hypercube bounds (Section 2): the axes span exactly these ranges
LHS_BOUNDS = {'R0V': (1, 5), 'R0Y': (1, 5), 'TV_years': (1, 10), 'TY_years': (1, 10), 'sigmaVY': (0, 1),
              'sigmaYV': (0, 1), 'm': (0.001, 0.01), 'season_phase': (0, 100), 'c_pandemic': (0.2, 0.8),
              'Lockdown_duration': (100, 600)}
# three ticks per axis, all inside the bounds
TICKS = {'R0V': (2, 3, 4), 'R0Y': (2, 3, 4), 'TV_years': (3, 6, 9), 'TY_years': (3, 6, 9),
         'c_pandemic': (0.3, 0.5, 0.7), 'Lockdown_duration': (200, 350, 500)}


def plot_parameter_space_new(df, name=None):
    """
    Takes the full DataFrame from the LHS sweep and plots the parameter spaces (Fig. 2).
    Plots each sample as a scatter point coloured by outcome:
      - green: timely Victoria recovery (Oct 2021 – Mar 2022)
      - yellow: early Victoria recovery (before Oct 2021)
      - red: late Victoria recovery (after Mar 2022)
    Drawn at its printed size (17.8 cm wide), so the font sizes below are the printed sizes;
    the axes span the Latin-hypercube bounds of each parameter. The four best fits by NH likelihood
    (Section 4) are marked as in the SI.
    """
    coexistence_target = (df['simple_coexist'] == True) & (df['recovery_Y'] != -1) & (df['recovery_V'] != -1)
    Yextinction_Vrecovery = coexistence_target & (df['extinct_Y'] == True) & (df['extinct_V'] == False) & (df['recovery_Y'] != -100)

    # --- Define Outcome Conditions ---
    base_req = coexistence_target & (df['extinct_Y'] == True) & (df['recovery_Y'] != -100)

    cond_green = Yextinction_Vrecovery & (df['recovery_V'] > 1.5) & (df['recovery_V'] <= 2)
    cond_early = base_req & (df['extinct_V'] == False) & (df['recovery_V'] <= 1.5) & (df['recovery_V'] != -1)
    cond_late  = base_req & (df['extinct_V'] == False) & (df['recovery_V'] > 2)

    df_green = df[cond_green]
    df_early = df[cond_early]
    df_late  = df[cond_late]
    top = df.loc[RANKED[:4]]  # ranks 1-4

    # Display labels
    display_labels = {
        'R0V': r"$R_{0,V}$",
        'R0Y': r"$R_{0,Y}$",
        'TV_years': r"$T_{V}$ (years)",
        'TY_years': r"$T_{Y}$ (years)",
        'sigmaVY': r"$\sigma_{VY}$",
        'sigmaYV': r"$\sigma_{YV}$",
        'm': r"$m$",
        'season_phase': r"$\phi$ (days)",
        'c_pandemic': r"$c_0$",
        'Lockdown_duration': r"$\tau$ (days)",
    }

    # ==========================================
    # PLOT
    # ==========================================
    print("Generating Categorized Parameter Space Subplots...")

    fig, axes = plt.subplots(1, 4, figsize=(7.0, 2.3))

    color_green = 'green'   # as the selected samples in the violin plots (Fig. S2)
    color_early = '#fee090'
    color_late  = '#d73027'

    plot_pairs = [
        ('R0Y', 'TY_years'),
        ('R0V', 'TV_years'),
        ('R0V', 'Lockdown_duration'),
        ('c_pandemic', 'Lockdown_duration')
    ]

    label_size, tick_size = 9, 7.5   # printed sizes, in points

    letters = []
    for i, (x_col, y_col) in enumerate(plot_pairs):
        ax = axes[i]
        letters.append(ax.text(-0.04, 1.03, ['A', 'B', 'C', 'D'][i], transform=ax.transAxes,
                               fontsize=11, fontname='Arial', fontweight='bold', va='bottom', ha='right'))

        # Plot early and late first (background), then green on top
        ax.scatter(df_early[x_col], df_early[y_col], c=color_early, s=1.2, alpha=0.5, edgecolor='none', rasterized=True)
        ax.scatter(df_late[x_col],  df_late[y_col],  c=color_late,  s=1.2, alpha=0.5, edgecolor='none', rasterized=True)
        ax.scatter(df_green[x_col], df_green[y_col], c=color_green, s=1.2, alpha=1, edgecolor='none', rasterized=True)
        for rank in (4, 3, 2, 1):  # rank 1 on top
            colour, marker, _ = TOP[rank]
            ax.scatter(top.iloc[rank - 1][x_col], top.iloc[rank - 1][y_col], s=28, c=colour, marker=marker,
                       edgecolor='white', linewidth=0.6, zorder=4)

        # Formatting
        ax.set_xlabel(display_labels[x_col], fontsize=label_size, labelpad=2)
        ax.set_ylabel(display_labels[y_col], fontsize=label_size, labelpad=2)
        ax.tick_params(labelsize=tick_size, length=2.5, pad=2)
        ax.set_xlim(LHS_BOUNDS[x_col])
        ax.set_ylim(LHS_BOUNDS[y_col])
        ax.set_xticks(TICKS[x_col])
        ax.set_yticks(TICKS[y_col])
        ax.set_box_aspect(1)

    # Legend
    legend_elements = [
        mlines.Line2D([], [], color=color_green, marker='o', linestyle='None',
                      markersize=5, alpha=1, label='Timely (Oct 2021 – Mar 2022)'),
        mlines.Line2D([], [], color=color_early, marker='o', linestyle='None',
                      markersize=5, alpha=1, label='Early (before Oct 2021)'),
        mlines.Line2D([], [], color=color_late,  marker='o', linestyle='None',
                      markersize=5, alpha=1, label='Late (after Mar 2022)'),
    ]
    best_fits = [mlines.Line2D([], [], color=colour, marker=marker, linestyle='None', markersize=6,
                               markeredgecolor='white', markeredgewidth=0.6,
                               label=f'Rank {rank}')  # the caption says rank 1 is the best fit
                 for rank, (colour, marker, _) in TOP.items()]

    # the best fits in a column at the right, centred on the panels; the outcomes just above the panel
    # letters (the saved figure is cropped to its contents)
    fits_legend = fig.legend(handles=best_fits, loc='center right', bbox_to_anchor=(1, 0.5), ncol=1, fontsize=8,
                             frameon=False, handletextpad=0.3, borderaxespad=0, labelspacing=0.7, title='Sample',
                             title_fontsize=8, alignment='left')
    fig.canvas.draw()
    to_fig = fig.transFigure.inverted()
    width = 1 - to_fig.transform(fits_legend.get_window_extent())[0, 0]
    fig.tight_layout(pad=0.4, rect=(0, 0, 1 - width, 1), w_pad=0.4)
    fig.canvas.draw()
    first = axes[0].get_position()
    fits_legend.set_bbox_to_anchor((1, (first.y0 + first.y1) / 2), transform=fig.transFigure)
    y = max(to_fig.transform(t.get_window_extent())[1, 1] for t in letters) + 0.01
    # 'Victoria reemergence:' to the left of the outcomes, the two centred together
    lead = fig.text(0, 0, 'Victoria reemergence:', fontsize=8, ha='right', va='center')
    outcomes = fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, y), ncol=3,
                          fontsize=8, frameon=False, handletextpad=0.2, columnspacing=1.2, borderpad=0.2)
    fig.canvas.draw()
    lead_width = np.diff(to_fig.transform(lead.get_window_extent())[:, 0])[0]
    outcomes.set_bbox_to_anchor((0.5 + lead_width / 2, y), transform=fig.transFigure)
    fig.canvas.draw()
    box = to_fig.transform(outcomes.get_window_extent())
    lead.set_position((box[0, 0], box[:, 1].mean()))

    if name is not None:
        name_parts = str(name).rsplit('.', 1)
        scatter_save_name = (f"{name_parts[0]}_densities.{name_parts[1]}"
                             if len(name_parts) == 2 else f"{name}_densities")
        plt.savefig(scatter_save_name, dpi=300, bbox_inches='tight')

    plt.show()


# Fig. 2: the selection outcomes in four parameter planes, with the four best fits (needs search_1M.csv)
if 'all_res' not in globals():
    all_res = pd.read_csv('search_1M.csv')
plot_parameter_space_new(all_res, 'evaluation.png')  # writes evaluation_densities.png

In [ ]:
# Figs. S20-S23: each of the four best fits against the NH data, drawn at their printed size in the SI
X_RANGE = (pd.Timestamp('2015-10-01'), pd.Timestamp('2025-09-30'))  # as in Fig. 1
SIZE = (7.2, 2.9)  # inches: the printed size in the SI, two figures per page
EVENTS = [  # vertical lines as in Fig. 3: date, label, colour, line style, label alignment (right: left of the line)
    (pd.Timestamp('2020-04-01'), 'lockdown', 'red', '--', 'left'),
    (pd.Timestamp('2020-07-01'), 'increased reporting', 'green', '--', 'left'),
]
# ranks 2 and 4: Victoria's minimum is weeks after Yamagata's extinction, so that label goes to the left of its line
YAM_LEFT = (2, 4)
LINEAGES = ['Victoria', 'Yamagata', 'Untyped']  # the artists are labelled 'Obs <lineage>', 'Pred <lineage>'


def plot_best_fit_nh(best_params_row, obs_dfs_list):
    """A sample's expected weekly reports against the NH data (reporting rates profiled out)."""
    nll, rho_dict, predictions_dict, diag = compute_nll_for_params_all_regions(best_params_row, obs_dfs_list)
    testing_cutoff = pd.to_datetime('2020-07-01')
    fig, ax = plt.subplots(1, 1, figsize=(15, 5))
    obs_df = obs_dfs_list[0]
    preds = predictions_dict[0]
    rho_pre, rho_post = rho_dict[0]
    muV, muY, tau_t = preds['muV'], preds['muY'], preds['tau_t']
    obsV, obsY, obsN = preds['obsV'], preds['obsY'], preds['obsN']
    x_dates = obs_df['ISO_WEEKSTARTDATE']
    pre_mask = x_dates < testing_cutoff
    muV_scaled = np.where(pre_mask, rho_pre * muV, rho_post * muV)
    muY_scaled = np.where(pre_mask, rho_pre * muY, rho_post * muY)
    obsV_pred = tau_t * muV_scaled
    obsY_pred = tau_t * muY_scaled
    obsN_pred = (1.0 - tau_t) * (muV_scaled + muY_scaled)
    ax.plot(x_dates, np.sqrt(obsV), 'o', label='Obs Victoria', alpha=0.6, markersize=4)
    ax.plot(x_dates, np.sqrt(obsY), 's', label='Obs Yamagata', alpha=0.6, markersize=4)
    ax.plot(x_dates, np.sqrt(obsN), '^', label='Obs Untyped', alpha=0.6, markersize=4)
    ax.plot(x_dates, np.sqrt(obsV_pred), '--', label='Pred Victoria', linewidth=2, alpha=0.8, color='C0')
    ax.plot(x_dates, np.sqrt(obsY_pred), '--', label='Pred Yamagata', linewidth=2, alpha=0.8, color='C1')
    ax.plot(x_dates, np.sqrt(obsN_pred), '--', label='Pred Untyped', linewidth=2, alpha=0.8, color='C2')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: f'{int(val**2):,}'))
    ax.set_ylabel('Weekly Reports (sqrt scale)', fontsize=11)
    ax.set_title(f'Northern Hemisphere ($\\rho_-$ = {rho_pre*100:.3f}%, $\\rho_+$ = {rho_post*100:.3f}%)',
                 fontsize=12, fontweight='bold')
    ax.legend(ncol=2, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('ISO Week Start Date', fontsize=11)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.set_xlim(x_dates.min(), x_dates.max())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.suptitle(f'Best-Fit Parameter Set (Global NLL={nll:.2f})', fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout()
    return fig


def restyle(fig, extinction, title=None, vic_minimum=None, yam_left=False):
    """The title = (text, marker, colour) above the plot; the x axis of Fig. 1 (a tick per year, unrotated
    labels, no axis label); solid lines for the predicted reports; vertical lines at the lockdown, the increase
    in reporting, Yamagata's predicted extinction and Victoria's predicted minimum (Yamagata's label to the left
    of its line with yam_left); and the printed size in the SI. Returns the reporting rates (the old title)."""
    fig.suptitle('').set_visible(False)
    fig.set_size_inches(*SIZE)
    for ax in fig.axes:
        rho = ax.get_title()
        ax.set_title('')
        ax.set_xlabel('')
        ax.set_ylabel(ax.get_ylabel(), fontsize=8.5)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.set_xlim(*X_RANGE)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, ha='center')
        ax.tick_params(labelsize=7.5)
        for line in ax.get_lines():
            line.set_markersize(2)
            if line.get_linestyle() == '--':  # the predicted reports
                line.set_linestyle('-')
                line.set_linewidth(1.2)
        events = EVENTS + [(extinction, 'Yam. extinction (predicted)', 'C1', '--', 'right' if yam_left else 'left')]
        if vic_minimum is not None:
            events.append((vic_minimum, 'Vic. minimum (predicted)', 'C0', '--', 'left'))
        for date, label, colour, style, side in events:
            ax.axvline(date, color=colour, linestyle=style, linewidth=1.2, alpha=0.7)
            # a label left of its line overlaps it with its descender space (the rotated text's right side)
            x = date + pd.Timedelta(days=10) if side == 'right' else date
            ax.text(x, 0.98, label, transform=ax.get_xaxis_transform(), color=colour, alpha=0.8, fontsize=7,
                    rotation=90, ha=side, va='top')
        # one row per lineage, "Victoria: (marker) observed (line) predicted": the legend fills columns first,
        # and the first column holds the lineage names with empty handles
        by_label = dict(zip(*ax.get_legend_handles_labels()[::-1]))
        blank = Line2D([], [], linestyle='none')
        handles = [blank] * len(LINEAGES) + [by_label[f'{k} {lin}'] for k in ('Obs', 'Pred') for lin in LINEAGES]
        labels = [f'{lin}:' for lin in LINEAGES] + ['observed'] * len(LINEAGES) + ['predicted'] * len(LINEAGES)
        legend = ax.legend(handles, labels, ncol=3, fontsize=7, handlelength=1.5, handletextpad=0.4, columnspacing=0.8)
        # no empty handle space before the lineage names (shrink the first column's empty handle boxes)
        for row in legend._legend_handle_box.get_children()[0].get_children():
            handle_box = row.get_children()[0]
            handle_box.width = 0
            row.sep = 0
    fig.tight_layout()
    if title is not None:  # just above the plot (the figure is saved cropped to its contents)
        text, marker, colour = title
        box = fig.axes[0].get_position()
        fig.legend(handles=[Line2D([0], [0], linestyle='none', marker=marker, markersize=6, color=colour,
                                   markeredgecolor='white', label=text)],
                   loc='lower center', bbox_to_anchor=((box.x0 + box.x1) / 2, box.y1), frameon=False,
                   handletextpad=0.3, borderaxespad=0.1, prop={'size': 8.5, 'weight': 'bold'})
    return rho


for rank in (1, 2, 3, 4):
    idx = RANKED[rank - 1]
    fig = plot_best_fit_nh(filtered.loc[idx], obs_dfs_list)
    rho = restyle(fig, pd.Timestamp(to_date(outputs.loc[idx, 't_yamagata_extinction'])), rank_title(rank),
                  pd.Timestamp(to_date(outputs.loc[idx, 't_min_prevalence_V'])), rank in YAM_LEFT)
    fig.savefig(f'nh_fit_rank{rank}.png', dpi=300, bbox_inches='tight')
    print(f'nh_fit_rank{rank}.png: sample {idx}, NH NLL {outputs.loc[idx, "NLL_nh"]:,.0f}; {rho}')
    plt.show()

In [ ]:
# Figs. S10-S11: the reporting rates of the filtered samples (no new simulations)
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from scipy.stats import spearmanr

AXIS_LABELS = {  # column: axis label (all ten parameters, in Table 1 order)
    'R0V': r'$R_{0,V}$', 'R0Y': r'$R_{0,Y}$', 'TV_years': r'$T_V$ (years)', 'TY_years': r'$T_Y$ (years)',
    'sigmaVY': r'$\sigma_{VY}$', 'sigmaYV': r'$\sigma_{YV}$', 'm': r'$m$',
    'season_phase': r'$\phi$ (days)', 'c_pandemic': r'$c_0$', 'Lockdown_duration': r'$\tau$ (days)',
}
SCATTER_PARAMS = ['R0V', 'R0Y', 'TV_years', 'TY_years', 'Lockdown_duration']
RHO = {'rho_pre': r'$\rho_{-}$', 'rho_post': r'$\rho_{+}$'}
RHO_LONG = {'rho_pre': r'$\rho_{-}$, before 1 July 2020 (%)', 'rho_post': r'$\rho_{+}$, from 1 July 2020 (%)'}


def panel_letter(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=11, fontweight='bold', color=INK,
            ha='right', va='bottom')


def summary(x, unit='%', fmt='{:.3f}'):
    lo, med, hi = np.percentile(x, [2.5, 50, 97.5])
    return f'median {fmt.format(med)}{unit}\n95%: {fmt.format(lo)}–{fmt.format(hi)}{unit}'


def histogram(ax, x, bins, top_values):
    """Bars plus a line for each of the best fits; returns the tallest bar so panels can share y-limits."""
    counts, _, _ = ax.hist(x, bins=bins, color=SAMPLES, edgecolor=SURFACE, linewidth=0.8)
    for rank, value in top_values.items():
        colour, _, pattern = TOP[rank]
        ax.axvline(value, color=colour, linestyle=pattern, linewidth=1.4, zorder=3)
    ax.set_ylabel('Samples')
    ax.grid(axis='x', visible=False)
    return counts.max()


def label_histogram(ax, text, text_left=False):
    x, ha = (0.03, 'left') if text_left else (0.97, 'right')
    ax.text(x, 1.0, text, transform=ax.transAxes, ha=ha, va='top', fontsize=7, color=INK2, zorder=4,
            bbox=dict(facecolor=SURFACE, edgecolor='none', boxstyle='square,pad=0.15'))


def scatter(ax, x, y, top_x, top_y, size=10, top_size=34):
    ax.scatter(x, y, s=size, color=SAMPLES, alpha=0.55, edgecolors=SURFACE, linewidths=0.3, zorder=2)
    for rank in reversed(list(top_x)):  # rank 1 on top
        colour, marker, _ = TOP[rank]
        ax.scatter([top_x[rank]], [top_y[rank]], s=top_size, color=colour, marker=marker, edgecolors=SURFACE,
                   linewidths=0.9, zorder=3)


def legend_handles(n, scatter=False):
    """With scatter=True, the samples as a dot and the best fits as their markers alone; otherwise the
    samples as a bar and the best fits as line and marker."""
    if scatter:
        return [Line2D([0], [0], linestyle='none', marker='o', markersize=4.5, color=SAMPLES, alpha=0.7,
                       markeredgecolor=SURFACE, markeredgewidth=0.3, label=f'Filtered samples (n = {n})')] + [
            Line2D([0], [0], linestyle='none', marker=marker, markersize=5.5, color=colour, markeredgecolor=SURFACE,
                   label=rank_label(rank)) for rank, (colour, marker, _) in TOP.items()]
    return [Patch(facecolor=SAMPLES, label=f'Filtered samples (n = {n})')] + [
        Line2D([0], [0], color=colour, linestyle=pattern, linewidth=1.4, marker=marker, markersize=5,
               markeredgecolor=SURFACE, label=rank_label(rank)) for rank, (colour, marker, pattern) in TOP.items()]


def legend_above(fig, n, scatter=False):
    """The legend just above the panels and their letters (save with bbox_inches='tight')."""
    fig.canvas.draw()
    top = max(ax.get_tightbbox().transformed(fig.transFigure.inverted()).y1 for ax in fig.axes)
    fig.legend(handles=legend_handles(n, scatter), loc='lower center', ncol=5, bbox_to_anchor=(0.5, top),
               borderaxespad=0.1, handlelength=1.2 if scatter else 2.6, handletextpad=0.5, columnspacing=1.5)


d = outputs
rho = d[list(RHO)] * 100  # percent of infections reported
ratio = d['rho_post'] / d['rho_pre']
tops = d[d['rank'].isin(list(TOP))].set_index('rank').sort_index()  # the four best fits
top_rho = tops[list(RHO)] * 100
top_ratio = tops['rho_post'] / tops['rho_pre']

corr = pd.DataFrame([{'parameter': p, **{f'spearman_{r}': spearmanr(d[p], d[r])[0] for r in RHO}} for p in AXIS_LABELS]
                    + [{'parameter': 'rho_post vs rho_pre', 'spearman_rho_pre': spearmanr(d['rho_pre'], d['rho_post'])[0],
                        'spearman_rho_post': np.nan}])
corr.to_csv('rho_spearman.csv', index=False, float_format='%.3f')

with plt.rc_context(STYLE):
    # Fig. S10: marginals, joint, ratio, in one row
    fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.5))
    short = {'rho_pre': '$\\rho_{-}$, before\n1 July 2020 (%)', 'rho_post': '$\\rho_{+}$, from\n1 July 2020 (%)'}
    top = float(rho.max().max()) * 1.04
    bins = np.linspace(0, top, 37)
    ymax = 1.3 * max(histogram(axes[0], rho['rho_pre'], bins, top_rho['rho_pre'].to_dict()),
                     histogram(axes[1], rho['rho_post'], bins, top_rho['rho_post'].to_dict()))
    for ax, r in zip(axes[:2], RHO):
        ax.set_xlim(0, top)
        ax.set_ylim(0, ymax)
        ax.xaxis.set_major_locator(MultipleLocator(0.02))
        ax.set_xlabel(short[r])
        label_histogram(ax, summary(rho[r]))
    ax = axes[2]
    ax.plot([0, top], [0, top], color=MUTED, linewidth=0.8, zorder=1)
    scatter(ax, rho['rho_pre'], rho['rho_post'], top_rho['rho_pre'].to_dict(), top_rho['rho_post'].to_dict())
    ax.set_xlim(0, top)
    ax.set_ylim(0, top)
    ax.set_aspect('equal', adjustable='box')
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(MultipleLocator(0.02))
    ax.text(0.76 * top, 0.76 * top, r'$\rho_{+} = \rho_{-}$', rotation=45, rotation_mode='anchor',
            transform_rotates_text=True, ha='left', va='top', fontsize=7, color=INK2, zorder=4)
    ax.set_xlabel(short['rho_pre'])
    ax.set_ylabel(short['rho_post'])
    r_joint = spearmanr(d['rho_pre'], d['rho_post'])[0]
    ax.text(0.97, 0.03, f'r = {r_joint:+.2f}\n' + r'$\rho_{+} > \rho_{-}$' + f' in {100 * (ratio > 1).mean():.0f}%',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=7, color=INK2)
    ax = axes[3]
    rbins = np.linspace(0, float(ratio.max()) * 1.04, 37)
    ax.axvline(1, color=MUTED, linewidth=0.8)
    rmax = histogram(ax, ratio, rbins, top_ratio.to_dict())
    ax.set_xlim(0, rbins[-1])
    ax.set_ylim(0, 1.3 * rmax)
    ax.set_xlabel(r'Ratio $\rho_{+} / \rho_{-}$')
    label_histogram(ax, summary(ratio, unit='', fmt='{:.2f}'), text_left=True)
    for ax, letter in zip(axes.flat, 'ABCD'):
        panel_letter(ax, letter)
    fig.tight_layout(w_pad=1.2)
    legend_above(fig, len(d))
    fig.savefig('rho_distributions.png', dpi=300, bbox_inches='tight', pad_inches=0.03)
    plt.show()

    # Fig. S11: the reporting rates against key parameters
    fig, axes = plt.subplots(2, len(SCATTER_PARAMS), figsize=(7.0, 3.7), sharex='col', sharey='row')
    for i, r in enumerate(RHO):
        for j, p in enumerate(SCATTER_PARAMS):
            ax = axes[i, j]
            scatter(ax, d[p], rho[r], tops[p].to_dict(), top_rho[r].to_dict(), size=6, top_size=26)
            ax.text(0.04, 0.96, f'r = {spearmanr(d[p], d[r])[0]:+.2f}', transform=ax.transAxes, ha='left', va='top',
                    fontsize=7, color=INK2, zorder=4,
                    bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.85, boxstyle='square,pad=0.15'))
            if i == 1:
                ax.set_xlabel(AXIS_LABELS[p])
            if j == 0:
                ax.set_ylabel(RHO_LONG[r].replace(', ', ',\n'))
    for ax, r in zip(axes[:, 0], RHO):  # headroom so the correlation labels clear every point
        ax.set_ylim(0, 1.22 * rho[r].max())
    fig.tight_layout(w_pad=0.8, h_pad=1.0)
    legend_above(fig, len(d), scatter=True)
    fig.savefig('rho_vs_parameters.png', dpi=300, bbox_inches='tight', pad_inches=0.03)
    plt.show()

print(corr.round(2).to_string(index=False))

## 5. Stochastic runs of the four best fits (Fig. 3, Figs. S13, S15, S17)

For each of the four best fits: `NSIMS` tau-leaping runs of Victoria from the moment Yamagata dies out (the
share in which Victoria dies out before its recovery is in Table S1), and `NSIMS` runs of a single Yamagata
infection introduced into the NH on each of the four dates (the probability of an outbreak of more than 100
infections is in Fig. 4 and Table S1). The model draws each figure (rank 1 is Fig. 3). About 40 minutes per
sample with $10^5$ runs.

In [ ]:
# ranks 2 and 4: Victoria's minimum is weeks after Yamagata's extinction, so that label goes to the left of its line
FIG3_STYLE = {2: {'labels_left': ('Yam. extinction',)}, 4: {'labels_left': ('Yam. extinction',)}}


def sto_figure(rank):
    return 'sto_nh.png' if rank == 1 else f'sto_nh_rank{rank}.png'


vic_rows, reintro_rows = [], []
for rank in (1, 2, 3, 4):
    idx = RANKED[rank - 1]
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        res = simulator.model(stochastic=True, nsims=NSIMS, stochastic2=True, nsims2=NSIMS, plot=True,
                              name=sto_figure(rank), title=rank_title(rank), style=FIG3_STYLE.get(rank),
                              **FIG3, **model_kwargs(filtered.loc[idx]))
    text = buf.getvalue()
    m = re.search(r'Victoria went extinct in ([\d.]+)%', text)
    vic_rows.append({'rank': rank, 'sample': idx, 'nsims': NSIMS,
                     'victoria_extinct_pct': float(m.group(1)) if m else float('nan'),
                     'min_prevalence_V_deterministic': res['min_prevalence_V'], 'recovery_V_years': res['recovery_V'],
                     'yamagata_extinction_date': to_date(res['t_yamagata_extinction']).date()})
    dates = re.findall(r'Hypothetical Introduction Date: (.+)', text)
    re_vals = [float(x) for x in re.findall(r'Calculated Initial Re: ([\d.]+)', text)]
    fade = [int(x) for x in re.findall(r'Stochastic Fade-outs \(< 100 cases\): (\d+) /', text)]
    reintro_rows += [{'rank': rank, 'sample': idx, 'date': d.strip(), 'R_e': r, 'nsims': NSIMS,
                      'P_outbreak_over_100': 1 - f / NSIMS} for d, r, f in zip(dates, re_vals, fade)]

stochastic_victoria = pd.DataFrame(vic_rows)
stochastic_victoria.to_csv('stochastic_victoria.csv', index=False)
reintroduction = pd.DataFrame(reintro_rows)
reintroduction.to_csv('reintroduction.csv', index=False)
print(stochastic_victoria.to_string(index=False))
print(reintroduction.to_string(index=False))

## 6. Yamagata's $R_e$ and outputs across the filtered samples (Fig. 4, Figs. S6–S9)

From `filtered_outputs.csv` (Section 4) and `reintroduction.csv` (Section 5); no new simulations.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# Fig. 4, drawn at its printed size (0.89 of a column, about 7.7 cm), so the font sizes below are
# the printed sizes. The R_e of the 595 filtered samples at each date are from Section 4.
labels = [
    "Reintroduction on 1 October 2026",
    "Reintroduction on 1 January 2027",
    "Reintroduction on 1 April 2027",
    "Reintroduction on 1 July 2027",
]
columns = ['Re_Oct1', 'Re_Jan1', 'Re_Apr1', 'Re_Jul1']
dates = ['Oct 1st 2026', 'Jan 1st 2027', 'Apr 1st 2027', 'Jul 1st 2027']
# The samples in the blue of the SI figures, and the four best fits to the NH data with their colours,
# markers and line patterns (Section 0); the probability that one reintroduction causes an outbreak of
# more than 100 infections (NSIMS stochastic runs each) is from Section 5.
ranked = RANKED
reintro = pd.read_csv('reintroduction.csv').set_index(['rank', 'date'])

saved = outputs.loc[filtered.index, columns]
data = [saved[c].to_numpy() for c in columns]
global_min = min(d.min() for d in data)
global_max = max(d.max() for d in data)
x = np.linspace(global_min, global_max, 500)
kdes = [gaussian_kde(d)(x) for d in data]
top = 1.1 * max(k.max() for k in kdes)

fig, axes = plt.subplots(nrows=4, ncols=1, sharex=True, figsize=(3.4, 6.0))
for i, ax in enumerate(axes):
    ax.text(-0.03, 1.04, ['A', 'B', 'C', 'D'][i], transform=ax.transAxes,
            fontsize=11, fontname='Arial', fontweight='bold', va='bottom', ha='right')
    ax.fill_between(x, kdes[i], color=SAMPLES, alpha=0.2, linewidth=0)
    ax.plot(x, kdes[i], color=SAMPLES, linewidth=1)
    # R_e = 1: solid and dark so it reads at print size
    ax.axvline(x=1, color='#404040', linestyle='-', linewidth=1.2, zorder=3)
    # The four best fits: a line from the axis up to the density curve, with the rank's marker on it
    handles = []
    for rank in reversed(list(TOP)):  # rank 1 drawn last, on top
        colour, marker, pattern = TOP[rank]
        Re = saved.loc[ranked[rank - 1], columns[i]]
        P = reintro.loc[(rank, dates[i]), 'P_outbreak_over_100']
        y = np.interp(Re, x, kdes[i])
        ax.plot([Re, Re], [0, y], color=colour, linestyle=pattern, linewidth=1.3, zorder=4)
        ax.plot(Re, y, marker=marker, color=colour, markersize=5, markeredgecolor='white', markeredgewidth=0.6,
                zorder=5)
        handles.insert(0, Line2D([0], [0], color=colour, linestyle=pattern, linewidth=1.3, marker=marker,
                                 markersize=5, markeredgecolor='white', markeredgewidth=0.6,
                                 label=f'Rank {rank}:  {Re:.2f},  {100 * P:.0f}%'))
    ax.legend(handles=handles, title=r'$R_e$,  P(outbreak > 100 cases)', loc='upper right', fontsize=6.5,
              title_fontsize=6.5, frameon=False, handlelength=2.6, borderaxespad=0.1, labelspacing=0.25,
              alignment='left')
    ax.set_title(labels[i], fontsize=8.5, fontweight='bold', loc='left', pad=3)
    ax.set_ylim(0, top)
    ax.set_yticks([])
    ax.set_ylabel('Density', fontsize=8)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
axes[-1].set_xlim(global_min, global_max)
axes[-1].tick_params(axis='x', labelsize=7.5)
axes[-1].set_xlabel("Effective reproduction number of Yamagata, $R_e$", fontsize=8)

plt.tight_layout(h_pad=0.6)
plt.savefig('Yam.png', dpi=300, bbox_inches='tight')  # bbox_inches ensures labels aren't cut off
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Fig. S6: R0,V - R0,Y against the NH negative log-likelihood, for the filtered samples (values from
# Section 4), in the style of the reporting-rate figures (Figs. S10-S11): Spearman's r, the
# four best fits with their colours and markers (legend in the empty top right), and at the right a box
# with the samples' median, interquartile range (box) and 95% range (whiskers), with their values.
x = outputs.loc[filtered.index, 'NLL_nh'].to_numpy() / 1e6
y = (filtered['R0V'] - filtered['R0Y']).to_numpy()
ranked = RANKED
position = {ix: k for k, ix in enumerate(filtered.index)}
r_s = spearmanr(x, y)[0]  # rank correlation: the same for y and log y
q = np.percentile(y, [2.5, 25, 50, 75, 97.5])

with plt.rc_context(STYLE):
    fig, (ax, axm) = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True,
                                  gridspec_kw={'width_ratios': [14, 1], 'wspace': 0.03})
    ax.scatter(x, y, s=12, color=SAMPLES, alpha=0.55, edgecolors=SURFACE, linewidths=0.3, zorder=2)
    for rank in reversed(list(TOP)):  # rank 1 on top
        colour, marker, _ = TOP[rank]
        k = position[ranked[rank - 1]]
        ax.scatter([x[k]], [y[k]], s=42, color=colour, marker=marker, edgecolors=SURFACE, linewidths=0.9, zorder=4)
    ax.set_xlabel('Negative log-likelihood of the NH data (millions)')
    ax.set_ylabel(r'$R_{0,V} - R_{0,Y}$')
    ax.text(0.015, 0.97, f'r = {r_s:+.2f}', transform=ax.transAxes, ha='left', va='top', fontsize=7,
            color=INK2, zorder=5,
            bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.85, boxstyle='square,pad=0.15'))

    # median, interquartile range and 95% range of the samples, with their values
    axm.boxplot(y, whis=(2.5, 97.5), showfliers=False, widths=0.6, patch_artist=True,
                boxprops=dict(facecolor=SAMPLES, alpha=0.35, edgecolor=INK2, linewidth=0.8),
                medianprops=dict(color='black', linewidth=1.2), whiskerprops=dict(color=INK2, linewidth=0.8),
                capprops=dict(color=INK2, linewidth=0.8))
    for v, name in zip(q, ['2.5%', '25%', 'median', '75%', '97.5%']):
        axm.text(1.42, v, f'{name} {v:.2f}', ha='left', va='center', fontsize=6.5, color=INK2, clip_on=False)
    axm.set_xlim(0.5, 1.4)
    axm.set_xticks([])
    axm.tick_params(axis='y', which='both', left=False)
    axm.grid(False)
    for side in ('left', 'bottom', 'top', 'right'):
        axm.spines[side].set_visible(False)

    handles = ([Line2D([0], [0], linestyle='none', marker='o', markersize=4.5, color=SAMPLES, alpha=0.7,
                       markeredgecolor=SURFACE, markeredgewidth=0.3, label=f'Filtered samples (n = {len(x)})')]
               + [Line2D([0], [0], linestyle='none', marker=m, markersize=5.5, color=c, markeredgecolor=SURFACE,
                         label=rank_label(r)) for r, (c, m, _) in TOP.items()])
    ax.legend(handles=handles, loc='upper right', fontsize=7, handletextpad=0.4, borderaxespad=0.5)  # empty there
    fig.subplots_adjust(left=0.085, right=0.89, bottom=0.14, top=0.97)
    fig.savefig('Yam_dif.png', dpi=300)
    plt.show()

In [ ]:
# Fig. S7: Victoria's minimum prevalence (deterministic, log scale) against R0,V - R0,Y, with dashed lines at 5
# and 10 infections and a line at R0,V = R0,Y; at the right, the samples' median, interquartile range (box) and
# 95% range (whiskers)
fil = filtered.copy()
fil['dR0'] = fil['R0V'] - fil['R0Y']
x, y = fil['dR0'].to_numpy(), fil['min_prevalence_V'].to_numpy()
r_s = spearmanr(x, y)[0]
q = np.percentile(y, [2.5, 25, 50, 75, 97.5])

with plt.rc_context(STYLE):
    fig, (ax, axm) = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True,
                                  gridspec_kw={'width_ratios': [14, 1], 'wspace': 0.03})
    ax.scatter(x, y, s=12, color=SAMPLES, alpha=0.55, edgecolors=SURFACE, linewidths=0.3, zorder=2)
    for rank in reversed(list(TOP)):  # rank 1 on top
        colour, marker, _ = TOP[rank]
        row = fil.loc[RANKED[rank - 1]]
        ax.scatter([row['dR0']], [row['min_prevalence_V']], s=42, color=colour, marker=marker, edgecolors=SURFACE,
                   linewidths=0.9, zorder=4)
    ax.set_yscale('log')
    for n in (5, 10):  # extinction thresholds of 5 and 10 infections
        ax.axhline(n, color=INK2, linewidth=0.8, linestyle=(0, (4, 2)), zorder=3)
        ax.text(ax.get_xlim()[1], n, f' {n} ', ha='right', va='bottom', fontsize=7, color=INK2, zorder=5)
    ax.axvline(0, color=MUTED, linewidth=0.8, zorder=1)
    ax.set_xlabel(r'$R_{0,V} - R_{0,Y}$')
    ax.set_ylabel('Minimum Victoria prevalence (log scale)')
    ax.text(0.015, 0.97, f'r = {r_s:+.2f}', transform=ax.transAxes, ha='left', va='top', fontsize=7, color=INK2,
            zorder=5, bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.85, boxstyle='square,pad=0.15'))
    axm.boxplot(y, whis=(2.5, 97.5), showfliers=False, widths=0.6, patch_artist=True,
                boxprops=dict(facecolor=SAMPLES, alpha=0.35, edgecolor=INK2, linewidth=0.8),
                medianprops=dict(color='black', linewidth=1.2), whiskerprops=dict(color=INK2, linewidth=0.8),
                capprops=dict(color=INK2, linewidth=0.8))
    for v, name in zip(q, ['2.5%', '25%', 'median', '75%', '97.5%']):
        axm.text(1.42, v, f'{name} {v:.3g}', ha='left', va='center', fontsize=6.5, color=INK2, clip_on=False)
    axm.set_xlim(0.5, 1.4)
    axm.set_xticks([])
    axm.tick_params(axis='y', which='both', left=False)
    axm.grid(False)
    for side in ('left', 'bottom', 'top', 'right'):
        axm.spines[side].set_visible(False)
    handles = ([Line2D([0], [0], linestyle='none', marker='o', markersize=4.5, color=SAMPLES, alpha=0.7,
                       markeredgecolor=SURFACE, markeredgewidth=0.3, label=f'Filtered samples (n = {len(fil)})')]
               + [Line2D([0], [0], linestyle='none', marker=m, markersize=5.5, color=c, markeredgecolor=SURFACE,
                         label=rank_label(r)) for r, (c, m, _) in TOP.items()])
    # the empty top left, below the correlation, on white so the R0,V = R0,Y line passes behind it
    legend = ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.0, 0.9), fontsize=7, handletextpad=0.4,
                       borderaxespad=0.5, frameon=True, facecolor=SURFACE, edgecolor='none', framealpha=1)
    legend.set_zorder(5)
    fig.subplots_adjust(left=0.085, right=0.89, bottom=0.14, top=0.97)
    fig.savefig('min_vs_r0_difference.png', dpi=300)
    plt.show()
print(f'{len(fil)} filtered samples; Spearman r = {r_s:+.2f}; median {q[2]:.3g}, interquartile range '
      f'{q[1]:.3g}-{q[3]:.3g}, 95% range {q[0]:.3g}-{q[4]:.3g}; R0,V < R0,Y in {(x < 0).sum()}; '
      f'R0,V > R0,Y in {100 * (x > 0).mean():.0f}% of the samples')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Fig. S8: peak 2026-27 Yamagata R_e against the NH negative log-likelihood, for the filtered samples (values from
# Section 4), in the style of the reporting-rate figures (Figs. S10-S11): Spearman's r, the
# four best fits with their colours and markers (legend in the empty top right), and at the right a box
# with the samples' median, interquartile range (box) and 95% range (whiskers), with their values.
x = outputs.loc[filtered.index, 'NLL_nh'].to_numpy() / 1e6
y = outputs.loc[filtered.index, 'R_eff_Y_peak_2026_27'].to_numpy()
ranked = RANKED
position = {ix: k for k, ix in enumerate(filtered.index)}
r_s = spearmanr(x, y)[0]  # rank correlation: the same for y and log y
q = np.percentile(y, [2.5, 25, 50, 75, 97.5])

with plt.rc_context(STYLE):
    fig, (ax, axm) = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True,
                                  gridspec_kw={'width_ratios': [14, 1], 'wspace': 0.03})
    ax.scatter(x, y, s=12, color=SAMPLES, alpha=0.55, edgecolors=SURFACE, linewidths=0.3, zorder=2)
    for rank in reversed(list(TOP)):  # rank 1 on top
        colour, marker, _ = TOP[rank]
        k = position[ranked[rank - 1]]
        ax.scatter([x[k]], [y[k]], s=42, color=colour, marker=marker, edgecolors=SURFACE, linewidths=0.9, zorder=4)
    ax.set_xlabel('Negative log-likelihood of the NH data (millions)')
    ax.set_ylabel(r'Peak 2026-27 Yamagata $R_e$')
    ax.text(0.015, 0.97, f'r = {r_s:+.2f}', transform=ax.transAxes, ha='left', va='top', fontsize=7,
            color=INK2, zorder=5,
            bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.85, boxstyle='square,pad=0.15'))

    # median, interquartile range and 95% range of the samples, with their values
    axm.boxplot(y, whis=(2.5, 97.5), showfliers=False, widths=0.6, patch_artist=True,
                boxprops=dict(facecolor=SAMPLES, alpha=0.35, edgecolor=INK2, linewidth=0.8),
                medianprops=dict(color='black', linewidth=1.2), whiskerprops=dict(color=INK2, linewidth=0.8),
                capprops=dict(color=INK2, linewidth=0.8))
    for v, name in zip(q, ['2.5%', '25%', 'median', '75%', '97.5%']):
        axm.text(1.42, v, f'{name} {v:.2f}', ha='left', va='center', fontsize=6.5, color=INK2, clip_on=False)
    axm.set_xlim(0.5, 1.4)
    axm.set_xticks([])
    axm.tick_params(axis='y', which='both', left=False)
    axm.grid(False)
    for side in ('left', 'bottom', 'top', 'right'):
        axm.spines[side].set_visible(False)

    handles = ([Line2D([0], [0], linestyle='none', marker='o', markersize=4.5, color=SAMPLES, alpha=0.7,
                       markeredgecolor=SURFACE, markeredgewidth=0.3, label=f'Filtered samples (n = {len(x)})')]
               + [Line2D([0], [0], linestyle='none', marker=m, markersize=5.5, color=c, markeredgecolor=SURFACE,
                         label=rank_label(r)) for r, (c, m, _) in TOP.items()])
    ax.legend(handles=handles, loc='upper right', fontsize=7, handletextpad=0.4, borderaxespad=0.5)  # empty there
    fig.subplots_adjust(left=0.085, right=0.89, bottom=0.14, top=0.97)
    fig.savefig('Yam_scatter.png', dpi=300)
    plt.show()

In [ ]:
# Fig. S9: the date of Victoria's minimum prevalence against the date of Yamagata's extinction (deterministic)
import matplotlib.dates as mdates

d = outputs.sort_values('rank').copy()
d['days_min_after_extinction'] = d['t_min_prevalence_V'] - d['t_yamagata_extinction']
x, y = d['t_yamagata_extinction'].to_numpy(), d['t_min_prevalence_V'].to_numpy()
r_s = spearmanr(x, y)[0]
after = (d['days_min_after_extinction'] > 0).sum()

with plt.rc_context(STYLE):
    fig, ax = plt.subplots(figsize=(7.2, 3.4))  # the printed size: two figures per page in the SI
    ax.scatter([to_date(t) for t in x], [to_date(t) for t in y], s=12, color=SAMPLES, alpha=0.55,
               edgecolors=SURFACE, linewidths=0.3, zorder=2)
    lo, hi = min(x.min(), y.min()) - 10, max(x.max(), y.max()) + 10
    ax.plot([to_date(lo), to_date(hi)], [to_date(lo), to_date(hi)], color=MUTED, linewidth=0.9,
            linestyle=(0, (4, 2)), zorder=1)
    t_label = y.max() + 2  # the axes end 15 days above the latest minimum
    ax.text(to_date(t_label), to_date(t_label - 16), 'y = x', rotation=45, rotation_mode='anchor',
            transform_rotates_text=True, ha='left', va='top', fontsize=7, color=INK2, zorder=4)
    ax.text(0.012, 0.97, f'r = {r_s:+.2f}', transform=ax.transAxes, ha='left', va='top', fontsize=7,
            color=INK2, zorder=5, bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.85,
                                            boxstyle='square,pad=0.15'))
    for rank in reversed(list(TOP)):  # rank 1 on top
        colour, marker, _ = TOP[rank]
        r = d[d['rank'] == rank].iloc[0]
        ax.scatter([to_date(r['t_yamagata_extinction'])], [to_date(r['t_min_prevalence_V'])], s=42, color=colour,
                   marker=marker, edgecolors=SURFACE, linewidths=0.9, zorder=4)
    ax.set_xlim(to_date(x.min() - 10), to_date(x.max() + 10))
    ax.set_ylim(to_date(y.min() - 15), to_date(y.max() + 15))
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(mdates.MonthLocator(interval=2))
        axis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', labelrotation=30)
    ax.set_xlabel("Date of Yamagata's extinction")
    ax.set_ylabel("Date of Victoria's minimum prevalence")
    handles = ([Line2D([0], [0], linestyle='none', marker='o', markersize=4.5, color=SAMPLES, alpha=0.7,
                       markeredgecolor=SURFACE, markeredgewidth=0.3, label=f'Filtered samples (n = {len(d)})')]
               + [Line2D([0], [0], linestyle='none', marker=m, markersize=5.5, color=c, markeredgecolor=SURFACE,
                         label=rank_label(rank)) for rank, (c, m, _) in TOP.items()])
    fig.tight_layout(rect=(0, 0, 0.79, 1))  # the legend goes in the right-hand fifth, added after the layout
    legend = ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=7, handletextpad=0.4,
                       borderaxespad=0.0)
    # below it, how long after Yamagata's extinction Victoria's minimum comes, across the samples
    gap = d['days_min_after_extinction'].to_numpy()
    q = np.percentile(gap, [2.5, 25, 50, 75, 97.5])
    fig.canvas.draw()
    below = ax.transAxes.inverted().transform(legend.get_window_extent())[0, 1]
    text = (f'median {q[2]:.0f} days\nIQR {q[1]:.0f} to {q[3]:.0f} days\n95%: {q[0]:.0f} to {q[4]:.0f} days'
            ).replace('-', '−')  # minus signs
    ax.text(1.01 + 0.012, below - 0.06, "Time gap between\nYamagata's extinction\nand Victoria's minimum:\n" + text,
            transform=ax.transAxes, ha='left', va='top', fontsize=7, color=INK2, linespacing=1.4)
    fig.savefig('extinction_vs_minimum.png', dpi=300)
    plt.show()
print(f'{len(d)} samples; minimum after extinction in {after}, before in {(d["days_min_after_extinction"] < 0).sum()}; '
      f'Spearman r = {r_s:.2f}')

In [ ]:
# Outputs across the samples (Results): median, 95% range, quartiles and where the best fit falls, treating
# the samples as equally weighted draws. The selected samples need target_runs.csv (Section 3).
def summarise(x, best, thresholds):
    x = x.dropna()
    q = np.percentile(x, [2.5, 25, 50, 75, 97.5])
    row = {'n': len(x), 'median': q[2], 'q2.5': q[0], 'q97.5': q[4], 'q25': q[1], 'q75': q[3],
           'mean': x.mean(), 'geomean': np.exp(np.log(x).mean()), 'min': x.min(), 'max': x.max()}
    row.update({f'pct_below_{t}': 100 * (x < t).mean() for t in thresholds})
    row.update({'best_fit': best, 'best_fit_percentile': 100 * (x < best).mean()})
    return row


sample_sets = {'filtered': outputs}
if os.path.exists('target_runs.csv'):
    sample_sets['selected'] = pd.read_csv('target_runs.csv', index_col=0)
THRESHOLDS = {'min_prevalence_V': [5, 10, 100, 1000], 'R_eff_Y_peak_2026_27': [1]}
rows = []
for name, samples in sample_sets.items():
    for out, thresholds in THRESHOLDS.items():
        if samples[out].isna().all():  # the peak R_e is computed for the filtered samples only
            continue
        rows.append({'output': out, 'samples': name,
                     **summarise(samples[out], samples.loc[RANKED[0], out], thresholds)})
summary_across_samples = pd.DataFrame(rows)
summary_across_samples.to_csv('summary_across_samples.csv', index=False)
summary_across_samples.set_index(['output', 'samples']).T

## 7. Leaky cross-immunity (SI: all-or-none immunity assumption; Figs. S12, S14, S16, S18)

Each filtered sample run again with leaky instead of all-or-none cross-immunity (`simulator_leaky.py`, reduced
susceptibility), checking the same selection rules; then the four best fits with leaky immunity, in the
layout of Fig. 3 (deterministic).

In [ ]:
def accepted(theta, res):
    """The biological filter plus the selection rules (count_and_plot)."""
    if not (theta['TV_years'] > theta['TY_years'] and theta['sigmaVY'] > theta['sigmaYV']):
        return False
    rv, ry = res['recovery_V'], res['recovery_Y']
    return (res['simple_coexist'] is True and res['extinct_Y'] is True and res['extinct_V'] is False
            and ry not in (-1, -100) and rv != -1 and 1.5 < rv <= 2)


def first_failed_rule(row):
    """The first selection rule a run breaks, in the order the model checks them."""
    if row['simple_coexist'] is not True and row['simple_coexist'] != 'True':
        return 'no coexistence before lockdown'
    if row['extinct_V'] in (True, 'True') or row['recovery_V'] in (-1, -10):
        return 'Victoria extinct'
    if row['recovery_Y'] == -1:
        return 'Yamagata extinct before lockdown'
    if row['extinct_Y'] in (False, 'False'):
        return 'Yamagata never extinct'
    if row['recovery_Y'] == -100:
        return 'Yamagata extinct only after Victoria re-emerged'
    return 'Victoria re-emergence outside 1.5-2 years after lockdown'


LEAKY = dict(leaky_mode='susceptibility', foi='balanced')
if os.path.exists('leaky_scan.csv'):
    leaky = pd.read_csv('leaky_scan.csv', index_col=0, keep_default_na=False, na_values=[''])
else:
    rows = {}
    with ProcessPoolExecutor(max_workers=N_CORES) as pool:
        pending = {pool.submit(simulator_leaky.model, plot=False, stochastic=False, **LEAKY, **model_kwargs(row)): idx
                   for idx, row in filtered.iterrows()}
        for future in tqdm(as_completed(pending), total=len(pending)):
            idx, res = pending.pop(future), future.result()
            ok = bool(accepted(filtered.loc[idx], res))
            rows[idx] = {'passes_leaky': ok, 'first_rule_broken': '' if ok else first_failed_rule(res),
                         'extinct_Y': res['extinct_Y'], 'extinct_V': res['extinct_V'],
                         'recovery_Y': res['recovery_Y'], 'recovery_V': res['recovery_V'],
                         'min_prevalence_V': res['min_prevalence_V']}
    leaky = pd.DataFrame.from_dict(rows, orient='index').loc[filtered.index]
    leaky.index.name = 'sample'
    leaky.to_csv('leaky_scan.csv')

passing = leaky['passes_leaky'].astype(str) == 'True'
fails = leaky[~passing]
print(f'With leaky cross-immunity, {int(passing.sum())} of the {len(leaky)} filtered samples still pass the selection '
      f'rules; {len(fails)} no longer do:')
for rule, n in fails['first_rule_broken'].value_counts().items():
    print(f'  {rule}: {n}')
vic = fails[fails['first_rule_broken'] == 'Victoria extinct']
print(f'  Victoria extinct: {int((outputs.loc[vic.index, "min_prevalence_V"] < 5).sum())} of {len(vic)} came within '
      '5 infections of extinction with all-or-none immunity')
ok = leaky[passing]
print(f'R0,V > R0,Y in {100 * (filtered.loc[ok.index, "R0V"] > filtered.loc[ok.index, "R0Y"]).mean():.0f}% of the samples '
      f'that still pass ({100 * (filtered["R0V"] > filtered["R0Y"]).mean():.0f}% of all filtered samples)')
shift = ok['recovery_V'].astype(float) - outputs.loc[ok.index, 'recovery_V']
factor = ok['min_prevalence_V'].astype(float) / outputs.loc[ok.index, 'min_prevalence_V']
print(f'Among them, Victoria re-emerges {shift.median() * 12:+.1f} months later (median); its minimum prevalence '
      f'changes by a median factor of {factor.median():.2f} (range {factor.min():.2f}-{factor.max():.2f})')

In [ ]:
# Figs. S12, S14, S16, S18: the four best fits with leaky cross-immunity, deterministic, in the layout of Fig. 3
for rank in (1, 2, 3, 4):
    idx = RANKED[rank - 1]
    quiet(simulator_leaky.model, plot=True, name=f'det_nh_leaky_rank{rank}.png',
          title=rank_title(rank, ', with leaky immunity'), style=FIG3_STYLE.get(rank),
          stochastic=False, stochastic2=True, nsims2=0, shortcuts=False, **FIG3, **LEAKY,
          **model_kwargs(filtered.loc[idx]))

## 8. Rank 3 without the lockdown (Fig. S19)

The third-best fit run on without the lockdown: its pre-2020 dynamics repeat every 12 years.

In [ ]:
RANK, YEARS, EVERY = 3, 24, 12  # the sample, the years drawn from 1 October 2015, a dashed line every EVERY years
NO_LOCKDOWN = 2.0  # Tpan: the lockdown at twice tspan, i.e. never
REGIONS = ['Northern Hemisphere', 'Tropics', 'Southern Hemisphere']

row = filtered.loc[RANKED[RANK - 1]]
t_end = T_REF_DAYS + YEARS * YEAR
res = quiet(simulator.model, plot=False, stochastic=False, shortcuts=False, tspan=t_end + 8, Tpan=NO_LOCKDOWN,
            **model_kwargs(row))
t, y, pops = res['t_eval'], res['y_all'], res['region_pops']

# NH infections per season (October-September), from the cumulative counters
starts = T_REF_DAYS + YEAR * np.arange(YEARS + 1)
print('NH infections per season, millions')
for lin, i in (('Victoria', 4), ('Yamagata', 5)):
    per_season = np.diff(np.interp(starts, t, y[i])) * pops[0] / 1e6
    print(f'{lin:9s} ' + ' '.join(f'{2015 + k}-{(2016 + k) % 100:02d}: {v:6.1f}' for k, v in enumerate(per_season)))

wt = res['weekly_times']
keep = (wt >= T_REF_DAYS) & (wt <= t_end)
dates = [to_date(x) for x in wt[keep]]
V, Y = res['weekly_infections_V'][:, keep] / 1e6, res['weekly_infections_Y'][:, keep] / 1e6
top = 1.3 * max(V.max(), Y.max())

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, sharey=True)
for r, ax in enumerate(axes):
    ax.plot(dates, V[r], color='tab:blue', lw=1.5, label='Victoria')
    ax.plot(dates, Y[r], color='tab:orange', lw=1.5, label='Yamagata')
    ax.set_yscale('function', functions=(np.sqrt, np.square))
    ax.set_ylim(0, top)
    ax.set_yticks([v for v in (0, 1, 4, 16, 36, 64, 100) if v < top])
    for n in range(1, YEARS // EVERY + (YEARS % EVERY > 0)):
        ax.axvline(to_date(T_REF_DAYS + n * EVERY * YEAR), color='0.25', ls='--', lw=1.2)
    ax.axvline(to_date(T_LOCK), color='0.6', ls=':', lw=1.2)
    ax.set_title(REGIONS[r], fontsize=12, fontweight='bold')
    ax.set_ylabel('weekly infections (millions)', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=9)
axes[1].legend(loc='upper left', fontsize=10, framealpha=1)  # the Tropics panel has room at the top
start = to_date(T_REF_DAYS)
for n in range(1, YEARS // EVERY + (YEARS % EVERY > 0)):
    axes[0].text(to_date(T_REF_DAYS + n * EVERY * YEAR - 40), 0.97 * top,
                 f'+{n * EVERY} years\n(from {start.day} {start:%b %Y})',
                 fontsize=10, color='0.25', va='top', ha='right', multialignment='right')
axes[0].text(to_date(T_LOCK + 40), 0.97 * top, '1 Apr 2020\n(no lockdown here)', fontsize=9, color='0.45', va='top')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_minor_locator(mdates.YearLocator(1))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].set_xlim(dates[0], dates[-1])
simulator.title_above(fig, rank_title(RANK, ', without lockdown'))
fig.savefig(f'no_lockdown_rank{RANK}.png', dpi=150)
plt.show()

## 9. The four best fits side by side (Table S1)

Influenza B infections per year are averaged over $n$ years, the period of the pre-pandemic (two-lineage)
attractor: with both lineages, without the lockdown, from 1 October 2015 (15.75 years after the start of
the simulation); with Victoria only, from the model as fitted (Yamagata dies out after the lockdown), from
1 January 2036 (15.75 years after the start of the lockdown), once Victoria has settled into its
single-lineage attractor. The cell checks the periods: the state comes back after $n$ years, and not after any
shorter whole number of years; the Victoria-only period is the smallest number of years after which
Victoria's state comes back to within 1%, and it divides $n$. Then the table, as `table_S1.csv` and as the
LaTeX rows of the SI table (`table_S1.tex`).

In [ ]:
PERIODS = {1: 4, 2: 2, 3: 12, 4: 1}  # years: the period of the pre-pandemic attractor, by rank
LATE = 20         # years after 1 January 2036 for the check that Victoria has settled
MAX_PERIOD = 12   # years: the longest Victoria-only period looked for
RETURN_TOL = 0.01
T_2015 = T_REF_DAYS               # 1 October 2015
T_V_ONLY = T_LOCK + 15.75 * YEAR  # 1 January 2036
CUM = {'V': [4, 10, 16], 'Y': [5, 11, 17]}  # cumulative new infections, by region
STATE = {'all': [0, 1, 2, 3, 6, 7, 8, 9, 12, 13, 14, 15], 'V': [0, 1, 6, 7, 12, 13]}  # S, I


def per_year(res, lineage, t0, years):
    """Global new infections of `lineage` from t0 over `years` years, in millions per year."""
    t, y, pops = res['t_eval'], res['y_all'], res['region_pops']
    t1 = t0 + years * YEAR
    assert t[0] <= t0 and t1 <= t[-1]
    return sum((np.interp(t1, t, y[i]) - np.interp(t0, t, y[i])) * N for i, N in zip(CUM[lineage], pops)) / years / 1e6


def returns(res, which, t0, years):
    """Largest relative difference between the state at t0 and k years later, for k = 1..years."""
    t, y = res['t_eval'], res['y_all']
    x = np.array([[np.interp(t0 + k * YEAR, t, y[i]) for i in STATE[which]] for k in range(years + 1)])
    return np.abs(x[1:] - x[0]).max(axis=1) / np.abs(x[0]).max()


rows = []
for rank, n in PERIODS.items():
    sample = RANKED[rank - 1]
    row = filtered.loc[sample]
    free = quiet(simulator.model, plot=False, stochastic=False, shortcuts=False, tspan=T_2015 + n * YEAR + 8,
                 Tpan=NO_LOCKDOWN, **model_kwargs(row))
    assert not free['extinct_V'] and not free['extinct_Y']
    back = returns(free, 'all', T_2015, n)
    tspan = T_V_ONLY + max(2 * n, LATE + n, MAX_PERIOD) * YEAR + 8
    fitted = quiet(simulator.model, plot=False, stochastic=False, shortcuts=False, tspan=tspan, Tpan=T_LOCK / tspan,
                   **model_kwargs(row))
    assert fitted['extinct_Y'] and not fitted['extinct_V']
    back_V = returns(fitted, 'V', T_V_ONLY, MAX_PERIOD)
    period_V = 1 + int(np.argmax(back_V < RETURN_TOL))
    assert back_V[period_V - 1] < RETURN_TOL and n % period_V == 0
    out = {'rank': rank, 'sample': sample, 'period_years': n, 'period_V_only_years': period_V,
           'V_two_lineages': per_year(free, 'V', T_2015, n), 'Y_two_lineages': per_year(free, 'Y', T_2015, n)}
    out['total_two_lineages'] = out['V_two_lineages'] + out['Y_two_lineages']
    out['V_only'] = per_year(fitted, 'V', T_V_ONLY, n)
    out['change_V_pct'] = 100 * (out['V_only'] / out['V_two_lineages'] - 1)
    out['change_total_pct'] = 100 * (out['V_only'] / out['total_two_lineages'] - 1)
    out['V_only_next_period'] = per_year(fitted, 'V', T_V_ONLY + n * YEAR, n)
    out[f'V_only_{2036 + LATE}'] = per_year(fitted, 'V', T_V_ONLY + LATE * YEAR, n)
    out['return_two_lineages'] = back[-1]
    out['closest_return_before_n'] = back[:-1].min() if n > 1 else np.nan
    out['return_V_only'] = back_V[period_V - 1]
    rows.append(out)
attractor_burden = pd.DataFrame(rows)
attractor_burden.to_csv('attractor_burden.csv', index=False)
attractor_burden.round(4)

In [ ]:
# Table S1
RANKS = [1, 2, 3, 4]
DATES = ['Oct 1st 2026', 'Jan 1st 2027', 'Apr 1st 2027', 'Jul 1st 2027']
DATE_LABELS = ['1 October 2026', '1 January 2027', '1 April 2027', '1 July 2027']
TABLE_PARAMS = [  # column, label, format (as in Table 1 of the main text)
    ('R0V', r'$R_{0,V}$', '{:.2f}'), ('R0Y', r'$R_{0,Y}$', '{:.2f}'),
    ('TV_years', r'$T_V$ (years)', '{:.2f}'), ('TY_years', r'$T_Y$ (years)', '{:.2f}'),
    ('sigmaVY', r'$\sigma_{VY}$', '{:.2f}'), ('sigmaYV', r'$\sigma_{YV}$', '{:.2f}'),
    ('m', r'$m$', '{:.4f}'), ('season_phase', r'$\phi$ (days)', '{:.1f}'),
    ('c_pandemic', r'$c_0$', '{:.2f}'), ('Lockdown_duration', r'$\tau$ (days)', '{:.0f}'),
]


def thousands(x):
    return f'{x:,.0f}'.replace(',', '{,}')


def signed_pct(x):
    return f'${"+" if x >= 0 else "-"}${abs(x):.1f}\\%'


def years(n):
    return f'{n:d} year' + ('' if n == 1 else 's')


def day_month_year(t):
    d = to_date(t)
    return f'{d.day} {d:%b %Y}'


top = outputs.loc[RANKED[:4]].assign(rank=RANKS).set_index('rank')
att = attractor_burden.set_index('rank').loc[RANKS]
vic = pd.read_csv('stochastic_victoria.csv').set_index('rank')
reintro = pd.read_csv('reintroduction.csv').set_index(['rank', 'date'])
rows = [  # (label, [rank 1, ..., rank 4]); None: a group heading
    ('Sample', [f'Rank {r}' for r in RANKS]),
    ('NH negative log-likelihood', [thousands(top.loc[r, 'NLL_nh']) for r in RANKS]),
    ('Parameters', None),
    *[(label, [fmt.format(top.loc[r, col]) for r in RANKS]) for col, label, fmt in TABLE_PARAMS],
    (r'Reporting rates (\%)', None),
    (r'$\rho_-$ (before 1 July 2020)', [f"{100 * top.loc[r, 'rho_pre']:.3f}" for r in RANKS]),
    (r'$\rho_+$ (from 1 July 2020)', [f"{100 * top.loc[r, 'rho_post']:.3f}" for r in RANKS]),
    ('After lockdown', None),
    ('Yamagata extinction', [day_month_year(top.loc[r, 't_yamagata_extinction']) for r in RANKS]),
    ('Minimum Victoria prevalence (date)',
     [f"{top.loc[r, 'min_prevalence_V']:.0f} ({day_month_year(top.loc[r, 't_min_prevalence_V'])})" for r in RANKS]),
    ('Victoria extinct in stochastic runs', [f"{vic.loc[r, 'victoria_extinct_pct']:.1f}\\%" for r in RANKS]),
    ('Yearly infections, averaged over one period of the attractor (millions)', None),
    ('Period of the pre-pandemic attractor', [years(att.loc[r, 'period_years']) for r in RANKS]),
    ('Period of the Victoria-only attractor', [years(att.loc[r, 'period_V_only_years']) for r in RANKS]),
    ('(i) Victoria, both lineages', [thousands(att.loc[r, 'V_two_lineages']) for r in RANKS]),
    ('(ii) Yamagata, both lineages', [thousands(att.loc[r, 'Y_two_lineages']) for r in RANKS]),
    ('(iii) total FluBV, both lineages', [thousands(att.loc[r, 'total_two_lineages']) for r in RANKS]),
    ('(iv) Victoria, without Yamagata', [thousands(att.loc[r, 'V_only']) for r in RANKS]),
    ('(v) change in Victoria infections', [signed_pct(att.loc[r, 'change_V_pct']) for r in RANKS]),
    ('(vi) change in total FluBV infections', [signed_pct(att.loc[r, 'change_total_pct']) for r in RANKS]),
    (r'Yamagata reintroduction: $R_e$ (probability of more than 100 infections)', None),
    ('2026--27 season peak (date)', [f"{top.loc[r, 'R_eff_Y_peak_2026_27']:.2f} "
                                     f"({day_month_year(top.loc[r, 't_R_eff_Y_peak_2026_27'])})" for r in RANKS]),
    *[(label, [f"{top.loc[r, col]:.2f} ({100 * reintro.loc[(r, d), 'P_outbreak_over_100']:.0f}\\%)" for r in RANKS])
      for d, col, label in zip(DATES, RE_COLUMNS, DATE_LABELS)],
]
lines = [r'\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lcccc@{}}', r'\toprule']
for k, (label, values) in enumerate(rows):
    if values is None:
        lines += [r'\midrule', r'\multicolumn{5}{@{}l}{\textit{' + label + r'}}\\']
    else:
        lines.append(f'{label} & ' + ' & '.join(values) + r'\\')
        if k == 0:
            lines.append(r'\midrule')
lines += [r'\bottomrule', r'\end{tabular*}']
with open('table_S1.tex', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines) + '\n')
table_S1 = pd.DataFrame({label: values for label, values in rows if values is not None}).T
table_S1.columns = table_S1.iloc[0]
table_S1 = table_S1.iloc[1:]
table_S1.to_csv('table_S1.csv')
table_S1

## 10. Victoria's re-emergence in the NH data (Materials and Methods)

The model declares Victoria re-emerged at the first week after its post-lockdown trough in which its NH weekly
infections reach 10% of the mean of its five pre-lockdown peaks (one per 365.25-day window ending on 1 April
2020). This applies the same rule to the NH FluNet data, with Victoria's weekly reports counted as the
Victoria-typed reports plus Victoria's share of the untyped B reports (that of Victoria among the typed
reports of the same week, as in the likelihood). Weeks with untyped but no typed reports take their season's
pooled share. Also printed: the 2021–22 peak relative to the mean peak, and the first week at other
thresholds.

In [ ]:
LOCK = pd.Timestamp('2020-04-01')
RATIO = outputs['rho_post'] / outputs['rho_pre']
RHO_RATIOS = {'median of the filtered samples': round(RATIO.median(), 2), 'best fit': round(RATIO[RANKED[0]], 2)}
THRESHOLDS_DATA = (0.02, 0.05, 0.10, 0.20, 0.30, 0.40)


def victoria_weekly():
    """Weekly NH Victoria reports plus Victoria's share of the untyped B reports, 1 Oct 2015 to 30 Sep 2025."""
    df = pd.read_csv('FluNet_NH.csv')
    df['week'] = pd.to_datetime(df['Week start date (ISO 8601 calendar)'])
    df = df[(df.week >= '2015-10-01') & (df.week < '2025-09-30')]
    w = df.groupby('week')[[BYAM, BVIC, BNOTDETERMINED]].sum().sort_index()
    w = w[w.sum(axis=1) > 0]
    typed = w[BVIC] + w[BYAM]
    season = np.where(w.index.month >= 10, w.index.year, w.index.year - 1)
    pooled = w[BVIC].groupby(season).transform('sum') / typed.groupby(season).transform('sum')
    share = (w[BVIC] / typed).where(typed > 0).fillna(pooled).fillna(1.0)
    return w[BVIC] + share * w[BNOTDETERMINED]


def describe(v, label, peaks):
    base = np.mean([p for _, p in peaks])
    print(f'\n{label}')
    for d, p in peaks:
        print(f'    peak {p:7,.0f}  (week of {d.date()})')
    s2122 = v[(v.index >= '2021-10-01') & (v.index < '2022-10-01')]
    ratio = s2122.max() / base
    print(f'  mean {base:,.0f}; 2021-22 peak {s2122.max():,.0f} (week of {s2122.idxmax().date()}) = {ratio:.1%} of it; '
          + ', '.join(f'{ratio / r:.1%} with rho+/rho- {r} ({k})' for k, r in RHO_RATIOS.items()))
    post = v[v.index >= LOCK]
    after = post[post.index > post.idxmin()]
    crossings = []
    for thr in THRESHOLDS_DATA:
        c = after[after >= thr * base]
        crossings.append(f"{thr:.0%} {c.index[0].date()} ({(c.index[0] - LOCK).days / 365.25:.2f} y)" if len(c)
                         else f'{thr:.0%} never')
    print('  first week at or above (selection window 1.5-2 years after lockdown, Oct 2021 to Mar 2022): '
          + '; '.join(crossings))


v = victoria_weekly()
windows = [(LOCK - pd.Timedelta(days=365.25 * (k + 1)), LOCK - pd.Timedelta(days=365.25 * k)) for k in range(4, -1, -1)]
describe(v, "The model's windows (365.25 days, ending 1 April 2020)",
         [(x.idxmax(), x.max()) for x in (v[(v.index >= a) & (v.index < b)] for a, b in windows)])
describe(v, 'October-September seasons, 2015-16 to 2019-20',
         [(x.idxmax(), x.max()) for x in (v[(v.index >= f'{yr}-10-01') & (v.index < min(pd.Timestamp(f'{yr + 1}-10-01'), LOCK))]
                                          for yr in range(2015, 2020))])